In [ ]:
# Colab cell 1

import os
import math
import time
import json
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Callable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import convolve2d
from scipy.optimize import minimize
from scipy.stats import ortho_group

# Reproducibility
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

# Output folders
ROOT_DIR = "/content/fixed_cnn_inverse_project"
FIG_DIR = os.path.join(ROOT_DIR, "figures")
CSV_DIR = os.path.join(ROOT_DIR, "csv")
NPY_DIR = os.path.join(ROOT_DIR, "arrays")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(NPY_DIR, exist_ok=True)

print("Project directory:", ROOT_DIR)

Project directory: /content/fixed_cnn_inverse_project


In [ ]:
# Colab cell 2

def set_seed(seed: int):
    np.random.seed(seed)
    random.seed(seed)

def save_fig(path: str, dpi: int = 200):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()

def sigmoid(z: np.ndarray, alpha: float, c: float) -> np.ndarray:
    # numerically stable sigmoid(alpha * (z-c))
    t = alpha * (z - c)
    t = np.clip(t, -60, 60)
    return 1.0 / (1.0 + np.exp(-t))

def sigmoid_prime_from_output(s: np.ndarray, alpha: float) -> np.ndarray:
    return alpha * s * (1.0 - s)

def sigmoid_double_prime_from_output(s: np.ndarray, alpha: float) -> np.ndarray:
    # h''(z) = alpha^2 * s(1-s)(1-2s)
    return (alpha ** 2) * s * (1.0 - s) * (1.0 - 2.0 * s)

def project_box(x: np.ndarray, low: float = 0.0, high: float = 1.0) -> np.ndarray:
    return np.clip(x, low, high)

def mse(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean((a - b) ** 2))

def binary_accuracy(pred: np.ndarray, target: np.ndarray, thr: float = 0.5) -> float:
    pb = (pred >= thr).astype(np.float64)
    tb = (target >= thr).astype(np.float64)
    return float(np.mean(pb == tb))

def iou_score(pred: np.ndarray, target: np.ndarray, thr: float = 0.5) -> float:
    pb = (pred >= thr).astype(np.uint8)
    tb = (target >= thr).astype(np.uint8)
    inter = np.logical_and(pb, tb).sum()
    union = np.logical_or(pb, tb).sum()
    if union == 0:
        return 1.0
    return float(inter / union)

def psnr(a: np.ndarray, b: np.ndarray, data_range: float = 1.0) -> float:
    err = np.mean((a - b) ** 2)
    if err <= 1e-12:
        return 100.0
    return float(20 * np.log10(data_range) - 10 * np.log10(err))

def rel_loss_reduction(loss0: float, lossT: float) -> float:
    if abs(loss0) < 1e-12:
        return 0.0
    return float((loss0 - lossT) / loss0)

def flatten_img(x: np.ndarray) -> np.ndarray:
    return x.reshape(-1)

def unflatten_img(x: np.ndarray, shape: Tuple[int, int]) -> np.ndarray:
    return x.reshape(shape)

In [ ]:
# Colab cell 3

def get_kernels() -> Dict[str, np.ndarray]:
    kernels = {}

    kernels["identity_like"] = np.array([
        [0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0]
    ], dtype=np.float64)

    kernels["avg_blur"] = (1.0 / 9.0) * np.ones((3, 3), dtype=np.float64)

    kernels["sobel_x"] = np.array([
        [-1.0, 0.0, 1.0],
        [-2.0, 0.0, 2.0],
        [-1.0, 0.0, 1.0]
    ], dtype=np.float64)

    kernels["sobel_y"] = np.array([
        [-1.0, -2.0, -1.0],
        [ 0.0,  0.0,  0.0],
        [ 1.0,  2.0,  1.0]
    ], dtype=np.float64)

    kernels["laplacian"] = np.array([
        [0.0, -1.0, 0.0],
        [-1.0, 4.0, -1.0],
        [0.0, -1.0, 0.0]
    ], dtype=np.float64)

    rng = np.random.default_rng(123)
    rand_k = rng.normal(size=(3, 3))
    rand_k /= (np.linalg.norm(rand_k) + 1e-12)
    kernels["random_norm"] = rand_k.astype(np.float64)

    return kernels

KERNELS = get_kernels()
print("Available kernels:", list(KERNELS.keys()))

Available kernels: ['identity_like', 'avg_blur', 'sobel_x', 'sobel_y', 'laplacian', 'random_norm']


In [ ]:
# Colab cell 4

def make_checkerboard(h: int, w: int, block: int = 8) -> np.ndarray:
    yy, xx = np.indices((h, w))
    return (((yy // block) + (xx // block)) % 2).astype(np.float64)

def make_vertical_stripes(h: int, w: int, stripe_w: int = 6) -> np.ndarray:
    _, xx = np.indices((h, w))
    return ((xx // stripe_w) % 2).astype(np.float64)

def make_horizontal_stripes(h: int, w: int, stripe_h: int = 6) -> np.ndarray:
    yy, _ = np.indices((h, w))
    return ((yy // stripe_h) % 2).astype(np.float64)

def make_circle(h: int, w: int, radius_ratio: float = 0.28) -> np.ndarray:
    yy, xx = np.indices((h, w))
    cy, cx = h / 2.0, w / 2.0
    r = min(h, w) * radius_ratio
    return ((((yy - cy) ** 2 + (xx - cx) ** 2) <= r ** 2)).astype(np.float64)

def make_sparse_dots(h: int, w: int, num_dots: int = 40, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    y = np.zeros((h, w), dtype=np.float64)
    idx = rng.choice(h * w, size=min(num_dots, h * w), replace=False)
    y.reshape(-1)[idx] = 1.0
    return y

def make_two_blobs(h: int, w: int) -> np.ndarray:
    yy, xx = np.indices((h, w))
    cy1, cx1 = int(0.35 * h), int(0.35 * w)
    cy2, cx2 = int(0.65 * h), int(0.65 * w)
    r1, r2 = int(0.14 * min(h, w)), int(0.12 * min(h, w))
    blob1 = ((yy - cy1) ** 2 + (xx - cx1) ** 2 <= r1 ** 2)
    blob2 = ((yy - cy2) ** 2 + (xx - cx2) ** 2 <= r2 ** 2)
    return np.logical_or(blob1, blob2).astype(np.float64)

def get_targets(h: int, w: int) -> Dict[str, np.ndarray]:
    return {
        "checkerboard": make_checkerboard(h, w, block=8),
        "vstripes": make_vertical_stripes(h, w, stripe_w=6),
        "hstripes": make_horizontal_stripes(h, w, stripe_h=6),
        "circle": make_circle(h, w, radius_ratio=0.28),
        "sparse_dots": make_sparse_dots(h, w, num_dots=40, seed=7),
        "two_blobs": make_two_blobs(h, w),
    }

TARGETS_64 = get_targets(64, 64)
print("Available targets:", list(TARGETS_64.keys()))

Available targets: ['checkerboard', 'vstripes', 'hstripes', 'circle', 'sparse_dots', 'two_blobs']


In [ ]:
# Colab cell 5

@dataclass
class ProblemConfig:
    image_shape: Tuple[int, int] = (64, 64)
    alpha: float = 10.0
    c: float = 0.5
    kernel_name: str = "sobel_x"
    target_name: str = "checkerboard"

class FixedCNNInverseProblem:
    def __init__(self, config: ProblemConfig, target: np.ndarray, kernel: np.ndarray):
        self.config = config
        self.image_shape = config.image_shape
        self.alpha = float(config.alpha)
        self.c = float(config.c)
        self.kernel_name = config.kernel_name
        self.target_name = config.target_name
        self.y = target.astype(np.float64)
        self.kernel = kernel.astype(np.float64)
        self.kernel_flip = np.flipud(np.fliplr(self.kernel))

        assert self.y.shape == self.image_shape, "Target shape mismatch."

    def conv(self, x: np.ndarray) -> np.ndarray:
        return convolve2d(x, self.kernel, mode="same", boundary="symm")

    def conv_transpose(self, z: np.ndarray) -> np.ndarray:
        # transpose of convolution under same symmetric boundary approximation
        return convolve2d(z, self.kernel_flip, mode="same", boundary="symm")

    def forward(self, x: np.ndarray) -> np.ndarray:
        ax = self.conv(x)
        return sigmoid(ax, alpha=self.alpha, c=self.c)

    def loss(self, x: np.ndarray) -> float:
        fx = self.forward(x)
        return float(np.sum((self.y - fx) ** 2))

    def grad(self, x: np.ndarray) -> np.ndarray:
        ax = self.conv(x)
        s = sigmoid(ax, alpha=self.alpha, c=self.c)
        sp = sigmoid_prime_from_output(s, alpha=self.alpha)
        # J(x)=||y-f(x)||^2 => grad = 2 A^T( (f-y) ⊙ h'(Ax) )
        residual = (s - self.y)
        g = 2.0 * self.conv_transpose(residual * sp)
        return g

    def grad_norm(self, x: np.ndarray) -> float:
        return float(np.linalg.norm(self.grad(x)))

    def metrics(self, x: np.ndarray) -> Dict[str, float]:
        fx = self.forward(x)
        return {
            "loss": self.loss(x),
            "grad_norm": self.grad_norm(x),
            "output_mse": mse(fx, self.y),
            "output_binary_acc": binary_accuracy(fx, self.y),
            "output_iou": iou_score(fx, self.y),
            "input_psnr_vs_target": psnr(x, self.y),
        }

In [ ]:
# Colab cell 6

def init_x(shape: Tuple[int, int], mode: str = "random", seed: int = 0, target: Optional[np.ndarray] = None) -> np.ndarray:
    rng = np.random.default_rng(seed)

    if mode == "random":
        x0 = rng.uniform(0.0, 1.0, size=shape)
    elif mode == "constant_half":
        x0 = np.full(shape, 0.5, dtype=np.float64)
    elif mode == "zeros":
        x0 = np.zeros(shape, dtype=np.float64)
    elif mode == "ones":
        x0 = np.ones(shape, dtype=np.float64)
    elif mode == "target_plus_noise":
        if target is None:
            raise ValueError("target is required for target_plus_noise initialization")
        noise = 0.15 * rng.normal(size=shape)
        x0 = project_box(target + noise)
    else:
        raise ValueError(f"Unknown init mode: {mode}")

    return x0.astype(np.float64)

In [ ]:
# Colab cell 7

def run_pgd(problem: FixedCNNInverseProblem,
            x0: np.ndarray,
            lr: float = 0.1,
            steps: int = 200) -> Dict:
    x = x0.copy()
    loss_hist, grad_hist = [], []
    t0 = time.time()

    for _ in range(steps):
        loss_hist.append(problem.loss(x))
        g = problem.grad(x)
        grad_hist.append(float(np.linalg.norm(g)))
        x = project_box(x - lr * g)

    elapsed = time.time() - t0
    return {
        "x_final": x,
        "loss_hist": loss_hist,
        "grad_hist": grad_hist,
        "time_sec": elapsed,
        "optimizer": "pgd"
    }

def run_momentum(problem: FixedCNNInverseProblem,
                 x0: np.ndarray,
                 lr: float = 0.05,
                 beta: float = 0.9,
                 steps: int = 200) -> Dict:
    x = x0.copy()
    v = np.zeros_like(x)
    loss_hist, grad_hist = [], []
    t0 = time.time()

    for _ in range(steps):
        loss_hist.append(problem.loss(x))
        g = problem.grad(x)
        grad_hist.append(float(np.linalg.norm(g)))
        v = beta * v - lr * g
        x = project_box(x + v)

    elapsed = time.time() - t0
    return {
        "x_final": x,
        "loss_hist": loss_hist,
        "grad_hist": grad_hist,
        "time_sec": elapsed,
        "optimizer": "momentum"
    }

def run_nesterov(problem: FixedCNNInverseProblem,
                 x0: np.ndarray,
                 lr: float = 0.05,
                 beta: float = 0.9,
                 steps: int = 200) -> Dict:
    x = x0.copy()
    v = np.zeros_like(x)
    loss_hist, grad_hist = [], []
    t0 = time.time()

    for _ in range(steps):
        loss_hist.append(problem.loss(x))
        g_look = problem.grad(project_box(x + beta * v))
        grad_hist.append(float(np.linalg.norm(g_look)))
        v = beta * v - lr * g_look
        x = project_box(x + v)

    elapsed = time.time() - t0
    return {
        "x_final": x,
        "loss_hist": loss_hist,
        "grad_hist": grad_hist,
        "time_sec": elapsed,
        "optimizer": "nesterov"
    }

def run_adam(problem: FixedCNNInverseProblem,
             x0: np.ndarray,
             lr: float = 0.03,
             beta1: float = 0.9,
             beta2: float = 0.999,
             eps: float = 1e-8,
             steps: int = 200) -> Dict:
    x = x0.copy()
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    loss_hist, grad_hist = [], []
    t0 = time.time()

    for t in range(1, steps + 1):
        loss_hist.append(problem.loss(x))
        g = problem.grad(x)
        grad_hist.append(float(np.linalg.norm(g)))

        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g ** 2)

        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)

        x = project_box(x - lr * m_hat / (np.sqrt(v_hat) + eps))

    elapsed = time.time() - t0
    return {
        "x_final": x,
        "loss_hist": loss_hist,
        "grad_hist": grad_hist,
        "time_sec": elapsed,
        "optimizer": "adam"
    }

def run_lbfgsb(problem: FixedCNNInverseProblem,
               x0: np.ndarray,
               maxiter: int = 200) -> Dict:
    shape = x0.shape
    x0_flat = x0.reshape(-1)
    bounds = [(0.0, 1.0)] * x0_flat.size
    loss_hist = []
    grad_hist = []
    t0 = time.time()

    def fun(z_flat):
        x = z_flat.reshape(shape)
        return problem.loss(x)

    def jac(z_flat):
        x = z_flat.reshape(shape)
        return problem.grad(x).reshape(-1)

    def callback(z_flat):
        x = z_flat.reshape(shape)
        loss_hist.append(problem.loss(x))
        grad_hist.append(problem.grad_norm(x))

    res = minimize(
        fun=fun,
        x0=x0_flat,
        jac=jac,
        method="L-BFGS-B",
        bounds=bounds,
        callback=callback,
        options={"maxiter": maxiter, "disp": False}
    )

    x_final = res.x.reshape(shape)
    elapsed = time.time() - t0

    if len(loss_hist) == 0:
        loss_hist = [problem.loss(x_final)]
        grad_hist = [problem.grad_norm(x_final)]

    return {
        "x_final": x_final,
        "loss_hist": loss_hist,
        "grad_hist": grad_hist,
        "time_sec": elapsed,
        "optimizer": "lbfgsb",
        "scipy_message": str(res.message),
        "scipy_success": bool(res.success),
        "scipy_nit": int(res.nit),
    }

OPTIMIZER_FUNCS = {
    "pgd": run_pgd,
    "momentum": run_momentum,
    "nesterov": run_nesterov,
    "adam": run_adam,
    "lbfgsb": run_lbfgsb,
}

In [ ]:
# Colab cell 8

def run_single_experiment(
    image_shape=(64, 64),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alpha=10.0,
    c=0.5,
    optimizer_name="pgd",
    init_mode="random",
    seed=0,
    optimizer_kwargs=None
):
    if optimizer_kwargs is None:
        optimizer_kwargs = {}

    targets = get_targets(*image_shape)
    target = targets[target_name]
    kernel = KERNELS[kernel_name]
    config = ProblemConfig(
        image_shape=image_shape,
        alpha=alpha,
        c=c,
        kernel_name=kernel_name,
        target_name=target_name
    )
    problem = FixedCNNInverseProblem(config=config, target=target, kernel=kernel)
    x0 = init_x(image_shape, mode=init_mode, seed=seed, target=target)

    result = OPTIMIZER_FUNCS[optimizer_name](problem, x0, **optimizer_kwargs)
    x_final = result["x_final"]

    metrics0 = problem.metrics(x0)
    metricsT = problem.metrics(x_final)

    summary = {
        "image_h": image_shape[0],
        "image_w": image_shape[1],
        "kernel_name": kernel_name,
        "target_name": target_name,
        "alpha": alpha,
        "c": c,
        "optimizer": optimizer_name,
        "init_mode": init_mode,
        "seed": seed,
        "time_sec": result["time_sec"],
        "loss_init": metrics0["loss"],
        "loss_final": metricsT["loss"],
        "grad_init": metrics0["grad_norm"],
        "grad_final": metricsT["grad_norm"],
        "output_mse_final": metricsT["output_mse"],
        "output_binary_acc_final": metricsT["output_binary_acc"],
        "output_iou_final": metricsT["output_iou"],
        "input_psnr_vs_target_final": metricsT["input_psnr_vs_target"],
        "rel_loss_reduction": rel_loss_reduction(metrics0["loss"], metricsT["loss"]),
        "num_iters_recorded": len(result["loss_hist"]),
    }

    if "scipy_message" in result:
        summary["scipy_message"] = result["scipy_message"]
        summary["scipy_success"] = result["scipy_success"]
        summary["scipy_nit"] = result["scipy_nit"]

    payload = {
        "problem": problem,
        "x0": x0,
        "x_final": x_final,
        "target": target,
        "result": result,
        "summary": summary,
    }
    return payload

In [ ]:
# Colab cell 9

payload = run_single_experiment(
    image_shape=(64, 64),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alpha=10.0,
    c=0.5,
    optimizer_name="pgd",
    init_mode="random",
    seed=0,
    optimizer_kwargs={"lr": 0.1, "steps": 150}
)

print(pd.Series(payload["summary"]))

image_h                                 64
image_w                                 64
kernel_name                        sobel_x
target_name                   checkerboard
alpha                                 10.0
c                                      0.5
optimizer                              pgd
init_mode                           random
seed                                     0
time_sec                          0.040961
loss_init                       1891.86902
loss_final                      703.838035
grad_init                       190.616256
grad_final                         17.9551
output_mse_final                  0.171835
output_binary_acc_final           0.827881
output_iou_final                   0.66948
input_psnr_vs_target_final        3.977302
rel_loss_reduction                0.627967
num_iters_recorded                     150
dtype: object


In [ ]:
# Colab cell 10

def plot_single_run(payload, save_prefix="single_run"):
    problem = payload["problem"]
    x0 = payload["x0"]
    x_final = payload["x_final"]
    target = payload["target"]
    result = payload["result"]
    summary = payload["summary"]

    fx0 = problem.forward(x0)
    fxT = problem.forward(x_final)

    # Convergence
    plt.figure(figsize=(6, 4))
    plt.plot(result["loss_hist"])
    plt.xlabel("Iteration")
    plt.ylabel("Loss J(x)")
    plt.title(
        f'{summary["optimizer"]} | alpha={summary["alpha"]}, c={summary["c"]}, '
        f'{summary["kernel_name"]}, {summary["target_name"]}'
    )
    save_fig(os.path.join(FIG_DIR, f"{save_prefix}_loss.png"))

    plt.figure(figsize=(6, 4))
    plt.plot(result["grad_hist"])
    plt.xlabel("Iteration")
    plt.ylabel("Gradient norm")
    plt.title(
        f'Gradient norm | {summary["optimizer"]} | alpha={summary["alpha"]}'
    )
    save_fig(os.path.join(FIG_DIR, f"{save_prefix}_grad.png"))

    # Images
    fig, axes = plt.subplots(2, 3, figsize=(10, 7))
    axes[0, 0].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[0, 0].set_title("Target y")
    axes[0, 1].imshow(x0, cmap="gray", vmin=0, vmax=1)
    axes[0, 1].set_title("Initial x0")
    axes[0, 2].imshow(x_final, cmap="gray", vmin=0, vmax=1)
    axes[0, 2].set_title("Final x")

    axes[1, 0].imshow(fx0, cmap="gray", vmin=0, vmax=1)
    axes[1, 0].set_title("f(x0)")
    axes[1, 1].imshow(fxT, cmap="gray", vmin=0, vmax=1)
    axes[1, 1].set_title("f(x_final)")
    axes[1, 2].imshow(np.abs(target - fxT), cmap="gray")
    axes[1, 2].set_title("|y - f(x_final)|")

    for ax in axes.ravel():
        ax.axis("off")

    save_fig(os.path.join(FIG_DIR, f"{save_prefix}_images.png"))

plot_single_run(payload, save_prefix="sanity_check")
print("Saved sanity-check plots to:", FIG_DIR)

Saved sanity-check plots to: /content/fixed_cnn_inverse_project/figures


In [ ]:
# Colab cell 11

def run_alpha_sweep(
    image_shape=(64, 64),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alphas=(1.0, 2.0, 5.0, 10.0, 20.0, 40.0),
    c=0.5,
    optimizers=None,
    init_mode="random",
    seeds=(0, 1, 2, 3, 4)
):
    if optimizers is None:
        optimizers = {
            "pgd": {"lr": 0.10, "steps": 200},
            "momentum": {"lr": 0.05, "beta": 0.9, "steps": 200},
            "nesterov": {"lr": 0.05, "beta": 0.9, "steps": 200},
            "adam": {"lr": 0.03, "steps": 200},
            "lbfgsb": {"maxiter": 200},
        }

    rows = []
    all_payloads = []

    for alpha in alphas:
        for opt_name, opt_kwargs in optimizers.items():
            for seed in seeds:
                payload = run_single_experiment(
                    image_shape=image_shape,
                    kernel_name=kernel_name,
                    target_name=target_name,
                    alpha=alpha,
                    c=c,
                    optimizer_name=opt_name,
                    init_mode=init_mode,
                    seed=seed,
                    optimizer_kwargs=opt_kwargs
                )
                rows.append(payload["summary"])
                all_payloads.append(payload)
                print(f"done alpha={alpha}, opt={opt_name}, seed={seed}")

    df = pd.DataFrame(rows)
    csv_path = os.path.join(CSV_DIR, "alpha_sweep_results.csv")
    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)
    return df, all_payloads

alpha_df, alpha_payloads = run_alpha_sweep()
alpha_df.head()

done alpha=1.0, opt=pgd, seed=0
done alpha=1.0, opt=pgd, seed=1
done alpha=1.0, opt=pgd, seed=2
done alpha=1.0, opt=pgd, seed=3
done alpha=1.0, opt=pgd, seed=4
done alpha=1.0, opt=momentum, seed=0
done alpha=1.0, opt=momentum, seed=1
done alpha=1.0, opt=momentum, seed=2
done alpha=1.0, opt=momentum, seed=3
done alpha=1.0, opt=momentum, seed=4
done alpha=1.0, opt=nesterov, seed=0
done alpha=1.0, opt=nesterov, seed=1
done alpha=1.0, opt=nesterov, seed=2
done alpha=1.0, opt=nesterov, seed=3
done alpha=1.0, opt=nesterov, seed=4
done alpha=1.0, opt=adam, seed=0
done alpha=1.0, opt=adam, seed=1
done alpha=1.0, opt=adam, seed=2
done alpha=1.0, opt=adam, seed=3
done alpha=1.0, opt=adam, seed=4
done alpha=1.0, opt=lbfgsb, seed=0
done alpha=1.0, opt=lbfgsb, seed=1
done alpha=1.0, opt=lbfgsb, seed=2
done alpha=1.0, opt=lbfgsb, seed=3
done alpha=1.0, opt=lbfgsb, seed=4


/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


done alpha=2.0, opt=pgd, seed=0
done alpha=2.0, opt=pgd, seed=1
done alpha=2.0, opt=pgd, seed=2
done alpha=2.0, opt=pgd, seed=3
done alpha=2.0, opt=pgd, seed=4
done alpha=2.0, opt=momentum, seed=0
done alpha=2.0, opt=momentum, seed=1
done alpha=2.0, opt=momentum, seed=2
done alpha=2.0, opt=momentum, seed=3
done alpha=2.0, opt=momentum, seed=4
done alpha=2.0, opt=nesterov, seed=0
done alpha=2.0, opt=nesterov, seed=1
done alpha=2.0, opt=nesterov, seed=2
done alpha=2.0, opt=nesterov, seed=3
done alpha=2.0, opt=nesterov, seed=4
done alpha=2.0, opt=adam, seed=0
done alpha=2.0, opt=adam, seed=1
done alpha=2.0, opt=adam, seed=2
done alpha=2.0, opt=adam, seed=3
done alpha=2.0, opt=adam, seed=4
done alpha=2.0, opt=lbfgsb, seed=0
done alpha=2.0, opt=lbfgsb, seed=1
done alpha=2.0, opt=lbfgsb, seed=2
done alpha=2.0, opt=lbfgsb, seed=3
done alpha=2.0, opt=lbfgsb, seed=4
done alpha=5.0, opt=pgd, seed=0
done alpha=5.0, opt=pgd, seed=1
done alpha=5.0, opt=pgd, seed=2
done alpha=5.0, opt=pgd, seed=3
do

,image_h,image_w,kernel_name,target_name,alpha,c,optimizer,init_mode,seed,time_sec,...,grad_final,output_mse_final,output_binary_acc_final,output_iou_final,input_psnr_vs_target_final,rel_loss_reduction,num_iters_recorded,scipy_message,scipy_success,scipy_nit
0,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,0,0.053199,...,28.269494,0.124104,0.898682,0.797363,4.534723,0.585968,200,NaN,NaN,NaN
1,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,1,0.052152,...,28.209253,0.125790,0.900146,0.800293,4.596249,0.591705,200,NaN,NaN,NaN
2,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,2,0.052189,...,28.223051,0.116074,0.931396,0.862793,4.591819,0.604143,200,NaN,NaN,NaN
3,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,3,0.052403,...,28.500860,0.121657,0.907715,0.815430,4.490099,0.591987,200,NaN,NaN,NaN
4,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,4,0.052381,...,28.289986,0.120188,0.902344,0.804688,4.408036,0.605142,200,NaN,NaN,NaN


In [ ]:
# Colab cell 12

def plot_alpha_sweep(df: pd.DataFrame):
    # mean final loss vs alpha
    plt.figure(figsize=(7, 5))
    for opt in sorted(df["optimizer"].unique()):
        sub = df[df["optimizer"] == opt]
        grp = sub.groupby("alpha", as_index=False)["loss_final"].mean().sort_values("alpha")
        plt.plot(grp["alpha"], grp["loss_final"], marker="o", label=opt)
    plt.xlabel("alpha")
    plt.ylabel("Mean final loss")
    plt.title("Alpha sensitivity: mean final loss")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "alpha_sweep_mean_final_loss.png"))

    # mean relative loss reduction
    plt.figure(figsize=(7, 5))
    for opt in sorted(df["optimizer"].unique()):
        sub = df[df["optimizer"] == opt]
        grp = sub.groupby("alpha", as_index=False)["rel_loss_reduction"].mean().sort_values("alpha")
        plt.plot(grp["alpha"], grp["rel_loss_reduction"], marker="o", label=opt)
    plt.xlabel("alpha")
    plt.ylabel("Mean relative loss reduction")
    plt.title("Alpha sensitivity: relative loss reduction")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "alpha_sweep_rel_loss_reduction.png"))

    # mean IoU
    plt.figure(figsize=(7, 5))
    for opt in sorted(df["optimizer"].unique()):
        sub = df[df["optimizer"] == opt]
        grp = sub.groupby("alpha", as_index=False)["output_iou_final"].mean().sort_values("alpha")
        plt.plot(grp["alpha"], grp["output_iou_final"], marker="o", label=opt)
    plt.xlabel("alpha")
    plt.ylabel("Mean output IoU")
    plt.title("Alpha sensitivity: mean IoU")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "alpha_sweep_iou.png"))

plot_alpha_sweep(alpha_df)
print("Alpha sweep plots saved.")

Alpha sweep plots saved.


In [ ]:
# Colab cell 13

def run_threshold_sweep(
    image_shape=(64, 64),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alpha=10.0,
    cs=(0.2, 0.4, 0.5, 0.6, 0.8),
    optimizers=None,
    init_mode="random",
    seeds=(0, 1, 2, 3, 4)
):
    if optimizers is None:
        optimizers = {
            "pgd": {"lr": 0.10, "steps": 200},
            "momentum": {"lr": 0.05, "beta": 0.9, "steps": 200},
            "adam": {"lr": 0.03, "steps": 200},
            "lbfgsb": {"maxiter": 200},
        }

    rows = []

    for c in cs:
        for opt_name, opt_kwargs in optimizers.items():
            for seed in seeds:
                payload = run_single_experiment(
                    image_shape=image_shape,
                    kernel_name=kernel_name,
                    target_name=target_name,
                    alpha=alpha,
                    c=c,
                    optimizer_name=opt_name,
                    init_mode=init_mode,
                    seed=seed,
                    optimizer_kwargs=opt_kwargs
                )
                rows.append(payload["summary"])
                print(f"done c={c}, opt={opt_name}, seed={seed}")

    df = pd.DataFrame(rows)
    csv_path = os.path.join(CSV_DIR, "threshold_sweep_results.csv")
    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)
    return df

threshold_df = run_threshold_sweep()
threshold_df.head()

done c=0.2, opt=pgd, seed=0
done c=0.2, opt=pgd, seed=1
done c=0.2, opt=pgd, seed=2
done c=0.2, opt=pgd, seed=3
done c=0.2, opt=pgd, seed=4
done c=0.2, opt=momentum, seed=0
done c=0.2, opt=momentum, seed=1
done c=0.2, opt=momentum, seed=2
done c=0.2, opt=momentum, seed=3
done c=0.2, opt=momentum, seed=4
done c=0.2, opt=adam, seed=0
done c=0.2, opt=adam, seed=1
done c=0.2, opt=adam, seed=2
done c=0.2, opt=adam, seed=3
done c=0.2, opt=adam, seed=4


/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


done c=0.2, opt=lbfgsb, seed=0
done c=0.2, opt=lbfgsb, seed=1
done c=0.2, opt=lbfgsb, seed=2
done c=0.2, opt=lbfgsb, seed=3
done c=0.2, opt=lbfgsb, seed=4
done c=0.4, opt=pgd, seed=0
done c=0.4, opt=pgd, seed=1
done c=0.4, opt=pgd, seed=2
done c=0.4, opt=pgd, seed=3
done c=0.4, opt=pgd, seed=4
done c=0.4, opt=momentum, seed=0
done c=0.4, opt=momentum, seed=1
done c=0.4, opt=momentum, seed=2
done c=0.4, opt=momentum, seed=3
done c=0.4, opt=momentum, seed=4
done c=0.4, opt=adam, seed=0
done c=0.4, opt=adam, seed=1
done c=0.4, opt=adam, seed=2
done c=0.4, opt=adam, seed=3
done c=0.4, opt=adam, seed=4
done c=0.4, opt=lbfgsb, seed=0
done c=0.4, opt=lbfgsb, seed=1
done c=0.4, opt=lbfgsb, seed=2
done c=0.4, opt=lbfgsb, seed=3
done c=0.4, opt=lbfgsb, seed=4
done c=0.5, opt=pgd, seed=0
done c=0.5, opt=pgd, seed=1
done c=0.5, opt=pgd, seed=2
done c=0.5, opt=pgd, seed=3
done c=0.5, opt=pgd, seed=4
done c=0.5, opt=momentum, seed=0
done c=0.5, opt=momentum, seed=1
done c=0.5, opt=momentum, seed=2
d

,image_h,image_w,kernel_name,target_name,alpha,c,optimizer,init_mode,seed,time_sec,...,grad_final,output_mse_final,output_binary_acc_final,output_iou_final,input_psnr_vs_target_final,rel_loss_reduction,num_iters_recorded,scipy_message,scipy_success,scipy_nit
0,64,64,sobel_x,checkerboard,10.0,0.2,pgd,random,0,0.052993,...,50.635880,0.194545,0.803223,0.650022,4.094945,0.574733,200,NaN,NaN,NaN
1,64,64,sobel_x,checkerboard,10.0,0.2,pgd,random,1,0.053607,...,46.615165,0.181860,0.816650,0.671335,4.120979,0.620485,200,NaN,NaN,NaN
2,64,64,sobel_x,checkerboard,10.0,0.2,pgd,random,2,0.053538,...,45.209159,0.175940,0.821777,0.675700,4.239696,0.605500,200,NaN,NaN,NaN
3,64,64,sobel_x,checkerboard,10.0,0.2,pgd,random,3,0.053877,...,26.769779,0.171481,0.828125,0.687389,4.187018,0.621561,200,NaN,NaN,NaN
4,64,64,sobel_x,checkerboard,10.0,0.2,pgd,random,4,0.053753,...,35.248156,0.179669,0.819336,0.672421,4.239888,0.612326,200,NaN,NaN,NaN


In [ ]:
# Colab cell 14

def plot_threshold_sweep(df: pd.DataFrame):
    plt.figure(figsize=(7, 5))
    for opt in sorted(df["optimizer"].unique()):
        sub = df[df["optimizer"] == opt]
        grp = sub.groupby("c", as_index=False)["loss_final"].mean().sort_values("c")
        plt.plot(grp["c"], grp["loss_final"], marker="o", label=opt)
    plt.xlabel("c")
    plt.ylabel("Mean final loss")
    plt.title("Threshold sensitivity: mean final loss")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "threshold_sweep_mean_final_loss.png"))

    plt.figure(figsize=(7, 5))
    for opt in sorted(df["optimizer"].unique()):
        sub = df[df["optimizer"] == opt]
        grp = sub.groupby("c", as_index=False)["output_iou_final"].mean().sort_values("c")
        plt.plot(grp["c"], grp["output_iou_final"], marker="o", label=opt)
    plt.xlabel("c")
    plt.ylabel("Mean IoU")
    plt.title("Threshold sensitivity: mean IoU")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "threshold_sweep_iou.png"))

plot_threshold_sweep(threshold_df)
print("Threshold sweep plots saved.")

Threshold sweep plots saved.


In [ ]:
# Colab cell 15

def run_kernel_sweep(
    image_shape=(64, 64),
    kernel_names=("identity_like", "avg_blur", "sobel_x", "laplacian", "random_norm"),
    target_name="checkerboard",
    alpha=10.0,
    c=0.5,
    optimizers=None,
    init_mode="random",
    seeds=(0, 1, 2, 3, 4)
):
    if optimizers is None:
        optimizers = {
            "pgd": {"lr": 0.10, "steps": 200},
            "momentum": {"lr": 0.05, "beta": 0.9, "steps": 200},
            "adam": {"lr": 0.03, "steps": 200},
            "lbfgsb": {"maxiter": 200},
        }

    rows = []

    for kernel_name in kernel_names:
        for opt_name, opt_kwargs in optimizers.items():
            for seed in seeds:
                payload = run_single_experiment(
                    image_shape=image_shape,
                    kernel_name=kernel_name,
                    target_name=target_name,
                    alpha=alpha,
                    c=c,
                    optimizer_name=opt_name,
                    init_mode=init_mode,
                    seed=seed,
                    optimizer_kwargs=opt_kwargs
                )
                rows.append(payload["summary"])
                print(f"done kernel={kernel_name}, opt={opt_name}, seed={seed}")

    df = pd.DataFrame(rows)
    csv_path = os.path.join(CSV_DIR, "kernel_sweep_results.csv")
    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)
    return df

kernel_df = run_kernel_sweep()
kernel_df.head()

done kernel=identity_like, opt=pgd, seed=0
done kernel=identity_like, opt=pgd, seed=1
done kernel=identity_like, opt=pgd, seed=2
done kernel=identity_like, opt=pgd, seed=3
done kernel=identity_like, opt=pgd, seed=4
done kernel=identity_like, opt=momentum, seed=0
done kernel=identity_like, opt=momentum, seed=1
done kernel=identity_like, opt=momentum, seed=2
done kernel=identity_like, opt=momentum, seed=3
done kernel=identity_like, opt=momentum, seed=4
done kernel=identity_like, opt=adam, seed=0
done kernel=identity_like, opt=adam, seed=1
done kernel=identity_like, opt=adam, seed=2
done kernel=identity_like, opt=adam, seed=3
done kernel=identity_like, opt=adam, seed=4
done kernel=identity_like, opt=lbfgsb, seed=0
done kernel=identity_like, opt=lbfgsb, seed=1
done kernel=identity_like, opt=lbfgsb, seed=2
done kernel=identity_like, opt=lbfgsb, seed=3
done kernel=identity_like, opt=lbfgsb, seed=4


/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


done kernel=avg_blur, opt=pgd, seed=0
done kernel=avg_blur, opt=pgd, seed=1
done kernel=avg_blur, opt=pgd, seed=2
done kernel=avg_blur, opt=pgd, seed=3
done kernel=avg_blur, opt=pgd, seed=4
done kernel=avg_blur, opt=momentum, seed=0
done kernel=avg_blur, opt=momentum, seed=1
done kernel=avg_blur, opt=momentum, seed=2
done kernel=avg_blur, opt=momentum, seed=3
done kernel=avg_blur, opt=momentum, seed=4
done kernel=avg_blur, opt=adam, seed=0
done kernel=avg_blur, opt=adam, seed=1
done kernel=avg_blur, opt=adam, seed=2
done kernel=avg_blur, opt=adam, seed=3
done kernel=avg_blur, opt=adam, seed=4
done kernel=avg_blur, opt=lbfgsb, seed=0
done kernel=avg_blur, opt=lbfgsb, seed=1
done kernel=avg_blur, opt=lbfgsb, seed=2
done kernel=avg_blur, opt=lbfgsb, seed=3
done kernel=avg_blur, opt=lbfgsb, seed=4
done kernel=sobel_x, opt=pgd, seed=0
done kernel=sobel_x, opt=pgd, seed=1
done kernel=sobel_x, opt=pgd, seed=2
done kernel=sobel_x, opt=pgd, seed=3
done kernel=sobel_x, opt=pgd, seed=4
done kerne

,image_h,image_w,kernel_name,target_name,alpha,c,optimizer,init_mode,seed,time_sec,...,grad_final,output_mse_final,output_binary_acc_final,output_iou_final,input_psnr_vs_target_final,rel_loss_reduction,num_iters_recorded,scipy_message,scipy_success,scipy_nit
0,64,64,identity_like,checkerboard,10.0,0.5,pgd,random,0,0.052545,...,0.153670,0.00012,1.0,1.0,26.045583,0.999693,200,NaN,NaN,NaN
1,64,64,identity_like,checkerboard,10.0,0.5,pgd,random,1,0.052575,...,0.154336,0.00012,1.0,1.0,26.003221,0.999703,200,NaN,NaN,NaN
2,64,64,identity_like,checkerboard,10.0,0.5,pgd,random,2,0.052637,...,0.154469,0.00012,1.0,1.0,25.993932,0.999704,200,NaN,NaN,NaN
3,64,64,identity_like,checkerboard,10.0,0.5,pgd,random,3,0.052781,...,0.153918,0.00012,1.0,1.0,26.031688,0.999700,200,NaN,NaN,NaN
4,64,64,identity_like,checkerboard,10.0,0.5,pgd,random,4,0.052621,...,0.153940,0.00012,1.0,1.0,26.030338,0.999702,200,NaN,NaN,NaN


In [ ]:
# Colab cell 16

def plot_kernel_sweep(df: pd.DataFrame):
    # final loss bar-like line plot
    summary = (
        df.groupby(["kernel_name", "optimizer"], as_index=False)
          .agg(loss_final_mean=("loss_final", "mean"),
               iou_mean=("output_iou_final", "mean"),
               rel_red_mean=("rel_loss_reduction", "mean"))
    )

    kernels = list(summary["kernel_name"].unique())
    opts = sorted(summary["optimizer"].unique())

    plt.figure(figsize=(9, 5))
    for opt in opts:
        sub = summary[summary["optimizer"] == opt]
        sub = sub.set_index("kernel_name").reindex(kernels).reset_index()
        plt.plot(sub["kernel_name"], sub["loss_final_mean"], marker="o", label=opt)
    plt.xticks(rotation=20)
    plt.ylabel("Mean final loss")
    plt.title("Kernel sensitivity: mean final loss")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "kernel_sweep_mean_final_loss.png"))

    plt.figure(figsize=(9, 5))
    for opt in opts:
        sub = summary[summary["optimizer"] == opt]
        sub = sub.set_index("kernel_name").reindex(kernels).reset_index()
        plt.plot(sub["kernel_name"], sub["iou_mean"], marker="o", label=opt)
    plt.xticks(rotation=20)
    plt.ylabel("Mean IoU")
    plt.title("Kernel sensitivity: mean IoU")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "kernel_sweep_iou.png"))

plot_kernel_sweep(kernel_df)
print("Kernel sweep plots saved.")

Kernel sweep plots saved.


In [ ]:
# Colab cell 17

def run_target_sweep(
    image_shape=(64, 64),
    kernel_name="sobel_x",
    target_names=("checkerboard", "vstripes", "circle", "sparse_dots", "two_blobs"),
    alpha=10.0,
    c=0.5,
    optimizers=None,
    init_mode="random",
    seeds=(0, 1, 2, 3, 4)
):
    if optimizers is None:
        optimizers = {
            "pgd": {"lr": 0.10, "steps": 200},
            "momentum": {"lr": 0.05, "beta": 0.9, "steps": 200},
            "adam": {"lr": 0.03, "steps": 200},
            "lbfgsb": {"maxiter": 200},
        }

    rows = []

    for target_name in target_names:
        for opt_name, opt_kwargs in optimizers.items():
            for seed in seeds:
                payload = run_single_experiment(
                    image_shape=image_shape,
                    kernel_name=kernel_name,
                    target_name=target_name,
                    alpha=alpha,
                    c=c,
                    optimizer_name=opt_name,
                    init_mode=init_mode,
                    seed=seed,
                    optimizer_kwargs=opt_kwargs
                )
                rows.append(payload["summary"])
                print(f"done target={target_name}, opt={opt_name}, seed={seed}")

    df = pd.DataFrame(rows)
    csv_path = os.path.join(CSV_DIR, "target_sweep_results.csv")
    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)
    return df

target_df = run_target_sweep()
target_df.head()

done target=checkerboard, opt=pgd, seed=0
done target=checkerboard, opt=pgd, seed=1
done target=checkerboard, opt=pgd, seed=2
done target=checkerboard, opt=pgd, seed=3
done target=checkerboard, opt=pgd, seed=4
done target=checkerboard, opt=momentum, seed=0
done target=checkerboard, opt=momentum, seed=1
done target=checkerboard, opt=momentum, seed=2
done target=checkerboard, opt=momentum, seed=3
done target=checkerboard, opt=momentum, seed=4
done target=checkerboard, opt=adam, seed=0
done target=checkerboard, opt=adam, seed=1
done target=checkerboard, opt=adam, seed=2
done target=checkerboard, opt=adam, seed=3
done target=checkerboard, opt=adam, seed=4


/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


done target=checkerboard, opt=lbfgsb, seed=0
done target=checkerboard, opt=lbfgsb, seed=1
done target=checkerboard, opt=lbfgsb, seed=2
done target=checkerboard, opt=lbfgsb, seed=3
done target=checkerboard, opt=lbfgsb, seed=4
done target=vstripes, opt=pgd, seed=0
done target=vstripes, opt=pgd, seed=1
done target=vstripes, opt=pgd, seed=2
done target=vstripes, opt=pgd, seed=3
done target=vstripes, opt=pgd, seed=4
done target=vstripes, opt=momentum, seed=0
done target=vstripes, opt=momentum, seed=1
done target=vstripes, opt=momentum, seed=2
done target=vstripes, opt=momentum, seed=3
done target=vstripes, opt=momentum, seed=4
done target=vstripes, opt=adam, seed=0
done target=vstripes, opt=adam, seed=1
done target=vstripes, opt=adam, seed=2
done target=vstripes, opt=adam, seed=3
done target=vstripes, opt=adam, seed=4
done target=vstripes, opt=lbfgsb, seed=0
done target=vstripes, opt=lbfgsb, seed=1
done target=vstripes, opt=lbfgsb, seed=2
done target=vstripes, opt=lbfgsb, seed=3
done target

,image_h,image_w,kernel_name,target_name,alpha,c,optimizer,init_mode,seed,time_sec,...,grad_final,output_mse_final,output_binary_acc_final,output_iou_final,input_psnr_vs_target_final,rel_loss_reduction,num_iters_recorded,scipy_message,scipy_success,scipy_nit
0,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,0,0.052614,...,31.162696,0.169797,0.829346,0.671523,3.962120,0.632380,200,NaN,NaN,NaN
1,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,1,0.052672,...,19.885457,0.172396,0.826904,0.671303,4.201831,0.644940,200,NaN,NaN,NaN
2,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,2,0.052664,...,16.116354,0.164373,0.835205,0.683247,4.215805,0.632062,200,NaN,NaN,NaN
3,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,3,0.051937,...,18.030477,0.177111,0.822510,0.662332,4.284332,0.611045,200,NaN,NaN,NaN
4,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,4,0.051905,...,12.509784,0.171592,0.828369,0.671188,4.028419,0.631532,200,NaN,NaN,NaN


In [ ]:
# Colab cell 18

def plot_target_sweep(df: pd.DataFrame):
    summary = (
        df.groupby(["target_name", "optimizer"], as_index=False)
          .agg(loss_final_mean=("loss_final", "mean"),
               iou_mean=("output_iou_final", "mean"))
    )

    targets = list(summary["target_name"].unique())
    opts = sorted(summary["optimizer"].unique())

    plt.figure(figsize=(9, 5))
    for opt in opts:
        sub = summary[summary["optimizer"] == opt]
        sub = sub.set_index("target_name").reindex(targets).reset_index()
        plt.plot(sub["target_name"], sub["loss_final_mean"], marker="o", label=opt)
    plt.xticks(rotation=20)
    plt.ylabel("Mean final loss")
    plt.title("Target sensitivity: mean final loss")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "target_sweep_mean_final_loss.png"))

    plt.figure(figsize=(9, 5))
    for opt in opts:
        sub = summary[summary["optimizer"] == opt]
        sub = sub.set_index("target_name").reindex(targets).reset_index()
        plt.plot(sub["target_name"], sub["iou_mean"], marker="o", label=opt)
    plt.xticks(rotation=20)
    plt.ylabel("Mean IoU")
    plt.title("Target sensitivity: mean IoU")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "target_sweep_iou.png"))

plot_target_sweep(target_df)
print("Target sweep plots saved.")

Target sweep plots saved.


In [ ]:
# Colab cell 19

def run_init_sweep(
    image_shape=(64, 64),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alpha=10.0,
    c=0.5,
    init_modes=("random", "constant_half", "zeros", "ones", "target_plus_noise"),
    optimizers=None,
    seeds=(0, 1, 2, 3, 4)
):
    if optimizers is None:
        optimizers = {
            "pgd": {"lr": 0.10, "steps": 200},
            "adam": {"lr": 0.03, "steps": 200},
            "lbfgsb": {"maxiter": 200},
        }

    rows = []

    for init_mode in init_modes:
        for opt_name, opt_kwargs in optimizers.items():
            for seed in seeds:
                payload = run_single_experiment(
                    image_shape=image_shape,
                    kernel_name=kernel_name,
                    target_name=target_name,
                    alpha=alpha,
                    c=c,
                    optimizer_name=opt_name,
                    init_mode=init_mode,
                    seed=seed,
                    optimizer_kwargs=opt_kwargs
                )
                rows.append(payload["summary"])
                print(f"done init={init_mode}, opt={opt_name}, seed={seed}")

    df = pd.DataFrame(rows)
    csv_path = os.path.join(CSV_DIR, "init_sweep_results.csv")
    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)
    return df

init_df = run_init_sweep()
init_df.head()

done init=random, opt=pgd, seed=0
done init=random, opt=pgd, seed=1
done init=random, opt=pgd, seed=2
done init=random, opt=pgd, seed=3
done init=random, opt=pgd, seed=4
done init=random, opt=adam, seed=0
done init=random, opt=adam, seed=1
done init=random, opt=adam, seed=2
done init=random, opt=adam, seed=3
done init=random, opt=adam, seed=4


/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


done init=random, opt=lbfgsb, seed=0
done init=random, opt=lbfgsb, seed=1
done init=random, opt=lbfgsb, seed=2
done init=random, opt=lbfgsb, seed=3
done init=random, opt=lbfgsb, seed=4
done init=constant_half, opt=pgd, seed=0
done init=constant_half, opt=pgd, seed=1
done init=constant_half, opt=pgd, seed=2
done init=constant_half, opt=pgd, seed=3
done init=constant_half, opt=pgd, seed=4
done init=constant_half, opt=adam, seed=0
done init=constant_half, opt=adam, seed=1
done init=constant_half, opt=adam, seed=2
done init=constant_half, opt=adam, seed=3
done init=constant_half, opt=adam, seed=4
done init=constant_half, opt=lbfgsb, seed=0
done init=constant_half, opt=lbfgsb, seed=1
done init=constant_half, opt=lbfgsb, seed=2
done init=constant_half, opt=lbfgsb, seed=3
done init=constant_half, opt=lbfgsb, seed=4
done init=zeros, opt=pgd, seed=0
done init=zeros, opt=pgd, seed=1
done init=zeros, opt=pgd, seed=2
done init=zeros, opt=pgd, seed=3
done init=zeros, opt=pgd, seed=4
done init=zeros

,image_h,image_w,kernel_name,target_name,alpha,c,optimizer,init_mode,seed,time_sec,...,grad_final,output_mse_final,output_binary_acc_final,output_iou_final,input_psnr_vs_target_final,rel_loss_reduction,num_iters_recorded,scipy_message,scipy_success,scipy_nit
0,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,0,0.052670,...,31.162696,0.169797,0.829346,0.671523,3.962120,0.632380,200,NaN,NaN,NaN
1,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,1,0.052066,...,19.885457,0.172396,0.826904,0.671303,4.201831,0.644940,200,NaN,NaN,NaN
2,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,2,0.051989,...,16.116354,0.164373,0.835205,0.683247,4.215805,0.632062,200,NaN,NaN,NaN
3,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,3,0.052127,...,18.030477,0.177111,0.822510,0.662332,4.284332,0.611045,200,NaN,NaN,NaN
4,64,64,sobel_x,checkerboard,10.0,0.5,pgd,random,4,0.052226,...,12.509784,0.171592,0.828369,0.671188,4.028419,0.631532,200,NaN,NaN,NaN


In [ ]:
# Colab cell 20

def plot_init_sweep(df: pd.DataFrame):
    summary = (
        df.groupby(["init_mode", "optimizer"], as_index=False)
          .agg(loss_final_mean=("loss_final", "mean"),
               iou_mean=("output_iou_final", "mean"))
    )

    init_modes = list(summary["init_mode"].unique())
    opts = sorted(summary["optimizer"].unique())

    plt.figure(figsize=(9, 5))
    for opt in opts:
        sub = summary[summary["optimizer"] == opt]
        sub = sub.set_index("init_mode").reindex(init_modes).reset_index()
        plt.plot(sub["init_mode"], sub["loss_final_mean"], marker="o", label=opt)
    plt.xticks(rotation=20)
    plt.ylabel("Mean final loss")
    plt.title("Initialization sensitivity: mean final loss")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "init_sweep_mean_final_loss.png"))

    plt.figure(figsize=(9, 5))
    for opt in opts:
        sub = summary[summary["optimizer"] == opt]
        sub = sub.set_index("init_mode").reindex(init_modes).reset_index()
        plt.plot(sub["init_mode"], sub["iou_mean"], marker="o", label=opt)
    plt.xticks(rotation=20)
    plt.ylabel("Mean IoU")
    plt.title("Initialization sensitivity: mean IoU")
    plt.legend()
    save_fig(os.path.join(FIG_DIR, "init_sweep_iou.png"))

plot_init_sweep(init_df)
print("Initialization sweep plots saved.")

Initialization sweep plots saved.


In [ ]:
# Colab cell 21

def orthonormal_directions(n: int, seed: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    u = rng.normal(size=n)
    u /= np.linalg.norm(u) + 1e-12

    v = rng.normal(size=n)
    v = v - np.dot(v, u) * u
    v /= np.linalg.norm(v) + 1e-12
    return u, v

def landscape_slice(problem: FixedCNNInverseProblem,
                    x_center: np.ndarray,
                    span: float = 1.0,
                    num: int = 81,
                    seed: int = 0):
    x_flat = x_center.reshape(-1)
    n = x_flat.size
    u, v = orthonormal_directions(n, seed=seed)

    a_vals = np.linspace(-span, span, num)
    b_vals = np.linspace(-span, span, num)
    Z = np.zeros((num, num), dtype=np.float64)

    for i, a in enumerate(a_vals):
        for j, b in enumerate(b_vals):
            z = x_flat + a * u + b * v
            z_img = project_box(z.reshape(problem.image_shape))
            Z[i, j] = problem.loss(z_img)

    return a_vals, b_vals, Z

def plot_landscape_slice(problem: FixedCNNInverseProblem,
                         x_center: np.ndarray,
                         title: str,
                         save_name: str,
                         span: float = 1.0,
                         num: int = 81,
                         seed: int = 0):
    a_vals, b_vals, Z = landscape_slice(problem, x_center, span=span, num=num, seed=seed)

    plt.figure(figsize=(6, 5))
    plt.contourf(b_vals, a_vals, Z, levels=40)
    plt.xlabel("Direction v coefficient")
    plt.ylabel("Direction u coefficient")
    plt.title(title)
    plt.colorbar(label="Loss J(x)")
    save_fig(os.path.join(FIG_DIR, save_name))

# Example landscape slice for low alpha
payload_low = run_single_experiment(
    image_shape=(32, 32),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alpha=2.0,
    c=0.5,
    optimizer_name="lbfgsb",
    init_mode="random",
    seed=0,
    optimizer_kwargs={"maxiter": 120}
)

plot_landscape_slice(
    problem=payload_low["problem"],
    x_center=payload_low["x_final"],
    title="Landscape slice near solution (alpha=2)",
    save_name="landscape_alpha_2.png",
    span=0.8,
    num=81,
    seed=0
)

# Example landscape slice for high alpha
payload_high = run_single_experiment(
    image_shape=(32, 32),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alpha=20.0,
    c=0.5,
    optimizer_name="lbfgsb",
    init_mode="random",
    seed=0,
    optimizer_kwargs={"maxiter": 120}
)

plot_landscape_slice(
    problem=payload_high["problem"],
    x_center=payload_high["x_final"],
    title="Landscape slice near solution (alpha=20)",
    save_name="landscape_alpha_20.png",
    span=0.8,
    num=81,
    seed=0
)

print("Landscape slice plots saved.")

/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


Landscape slice plots saved.


In [ ]:
# Colab cell 22

def collect_reconstruction_examples():
    examples = []
    settings = [
        ("checkerboard", "sobel_x", 2.0),
        ("checkerboard", "sobel_x", 10.0),
        ("checkerboard", "sobel_x", 20.0),
        ("circle", "laplacian", 10.0),
        ("two_blobs", "avg_blur", 10.0),
        ("sparse_dots", "random_norm", 10.0),
    ]

    for target_name, kernel_name, alpha in settings:
        payload = run_single_experiment(
            image_shape=(64, 64),
            kernel_name=kernel_name,
            target_name=target_name,
            alpha=alpha,
            c=0.5,
            optimizer_name="lbfgsb",
            init_mode="random",
            seed=0,
            optimizer_kwargs={"maxiter": 200}
        )
        examples.append(payload)
    return examples

examples = collect_reconstruction_examples()

fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))

for i, payload in enumerate(examples):
    target = payload["target"]
    x_final = payload["x_final"]
    fx = payload["problem"].forward(x_final)
    s = payload["summary"]

    axes[i, 0].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[i, 0].set_title(f'Target\n{s["target_name"]}')
    axes[i, 1].imshow(x_final, cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title(f'Reconstructed x\nopt={s["optimizer"]}')
    axes[i, 2].imshow(fx, cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title(f'f(x), alpha={s["alpha"]}\nkernel={s["kernel_name"]}')

    for j in range(3):
        axes[i, j].axis("off")

save_fig(os.path.join(FIG_DIR, "representative_reconstructions.png"))
print("Representative reconstructions figure saved.")

/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


Representative reconstructions figure saved.


In [ ]:
# Colab cell 23

def summarize_for_paper(df: pd.DataFrame, group_cols: List[str], metric_cols: List[str]) -> pd.DataFrame:
    agg_dict = {}
    for m in metric_cols:
        agg_dict[m] = ["mean", "std"]

    out = df.groupby(group_cols).agg(agg_dict)
    out.columns = ["_".join(col).strip() for col in out.columns.values]
    out = out.reset_index()
    return out

paper_alpha_table = summarize_for_paper(
    alpha_df,
    group_cols=["alpha", "optimizer"],
    metric_cols=["loss_final", "rel_loss_reduction", "output_iou_final", "time_sec"]
)
paper_alpha_table.to_csv(os.path.join(CSV_DIR, "paper_alpha_table.csv"), index=False)

paper_kernel_table = summarize_for_paper(
    kernel_df,
    group_cols=["kernel_name", "optimizer"],
    metric_cols=["loss_final", "rel_loss_reduction", "output_iou_final", "time_sec"]
)
paper_kernel_table.to_csv(os.path.join(CSV_DIR, "paper_kernel_table.csv"), index=False)

paper_threshold_table = summarize_for_paper(
    threshold_df,
    group_cols=["c", "optimizer"],
    metric_cols=["loss_final", "rel_loss_reduction", "output_iou_final", "time_sec"]
)
paper_threshold_table.to_csv(os.path.join(CSV_DIR, "paper_threshold_table.csv"), index=False)

print("Saved paper summary tables.")
paper_alpha_table.head(10)

Saved paper summary tables.


,alpha,optimizer,loss_final_mean,loss_final_std,rel_loss_reduction_mean,rel_loss_reduction_std,output_iou_final_mean,output_iou_final_std,time_sec_mean,time_sec_std
0,1.0,adam,504.871382,9.907335,0.590062,0.007996,0.818652,0.017389,0.055757,0.000258
1,1.0,lbfgsb,493.791753,11.485152,0.599056,0.009284,0.833496,0.008625,0.037638,0.007710
2,1.0,momentum,498.780852,15.247167,0.594987,0.013038,0.829980,0.011634,0.052703,0.000158
3,1.0,nesterov,499.210898,13.613814,0.594691,0.009306,0.821582,0.009830,0.054149,0.000120
4,1.0,pgd,497.920942,15.375055,0.595789,0.008439,0.816113,0.026981,0.052465,0.000425
5,2.0,adam,337.241820,13.462332,0.771109,0.006736,0.871973,0.014043,0.055533,0.000116
6,2.0,lbfgsb,336.950411,12.048741,0.771303,0.005411,0.869434,0.005209,0.081085,0.019232
7,2.0,momentum,328.460460,7.875516,0.777016,0.004651,0.870130,0.006678,0.052429,0.000136
8,2.0,nesterov,322.782527,10.087649,0.780900,0.004892,0.869751,0.007406,0.053888,0.000083
9,2.0,pgd,431.153307,7.438253,0.707257,0.006654,0.679255,0.004378,0.052123,0.000192


In [ ]:
# Colab cell 24

def export_best_runs(df: pd.DataFrame, label: str):
    idx = df.groupby(["optimizer"])["loss_final"].idxmin()
    best = df.loc[idx].sort_values("loss_final").reset_index(drop=True)
    path = os.path.join(CSV_DIR, f"best_runs_{label}.csv")
    best.to_csv(path, index=False)
    print("Saved:", path)
    return best

best_alpha_runs = export_best_runs(alpha_df, "alpha")
best_kernel_runs = export_best_runs(kernel_df, "kernel")
best_threshold_runs = export_best_runs(threshold_df, "threshold")

best_alpha_runs

Saved: /content/fixed_cnn_inverse_project/csv/best_runs_alpha.csv
Saved: /content/fixed_cnn_inverse_project/csv/best_runs_kernel.csv
Saved: /content/fixed_cnn_inverse_project/csv/best_runs_threshold.csv


,image_h,image_w,kernel_name,target_name,alpha,c,optimizer,init_mode,seed,time_sec,...,grad_final,output_mse_final,output_binary_acc_final,output_iou_final,input_psnr_vs_target_final,rel_loss_reduction,num_iters_recorded,scipy_message,scipy_success,scipy_nit
0,64,64,sobel_x,checkerboard,2.0,0.5,nesterov,random,3,0.053851,...,29.664269,0.075342,0.937744,0.875488,4.348634,0.787725,200,NaN,NaN,NaN
1,64,64,sobel_x,checkerboard,2.0,0.5,adam,random,3,0.055499,...,31.319824,0.077392,0.943848,0.887695,4.332525,0.781948,200,NaN,NaN,NaN
2,64,64,sobel_x,checkerboard,2.0,0.5,momentum,random,3,0.052653,...,29.976021,0.077641,0.937744,0.875488,4.358191,0.781247,200,NaN,NaN,NaN
3,64,64,sobel_x,checkerboard,2.0,0.5,lbfgsb,random,3,0.113051,...,29.426211,0.079740,0.935059,0.870117,4.537278,0.775334,78,ABNORMAL:,False,78.0
4,64,64,sobel_x,checkerboard,2.0,0.5,pgd,random,3,0.052038,...,76.284033,0.103499,0.836426,0.673171,4.450169,0.708394,200,NaN,NaN,NaN


In [ ]:
# Colab cell 25

def finite_difference_grad_check(problem: FixedCNNInverseProblem,
                                 x: np.ndarray,
                                 eps: float = 1e-6,
                                 num_coords: int = 20,
                                 seed: int = 0):
    rng = np.random.default_rng(seed)
    x_flat = x.reshape(-1).copy()
    g_analytic = problem.grad(x).reshape(-1)

    idxs = rng.choice(x_flat.size, size=min(num_coords, x_flat.size), replace=False)
    errs = []

    for idx in idxs:
        xp = x_flat.copy()
        xm = x_flat.copy()
        xp[idx] += eps
        xm[idx] -= eps
        xp_img = project_box(xp.reshape(problem.image_shape))
        xm_img = project_box(xm.reshape(problem.image_shape))

        fd = (problem.loss(xp_img) - problem.loss(xm_img)) / (2 * eps)
        err = abs(fd - g_analytic[idx])
        errs.append(err)

    return {
        "mean_abs_error": float(np.mean(errs)),
        "max_abs_error": float(np.max(errs)),
        "num_coords_checked": len(errs),
    }

targets = get_targets(32, 32)
cfg = ProblemConfig(image_shape=(32, 32), alpha=5.0, c=0.5, kernel_name="sobel_x", target_name="checkerboard")
problem = FixedCNNInverseProblem(cfg, targets["checkerboard"], KERNELS["sobel_x"])
x_test = init_x((32, 32), mode="random", seed=0)
check = finite_difference_grad_check(problem, x_test)
print(check)

{'mean_abs_error': 0.5452151964805216, 'max_abs_error': 4.955849295567874, 'num_coords_checked': 20}


In [ ]:
# Colab cell 26

print("Figures saved in:", FIG_DIR)
print("CSV files saved in:", CSV_DIR)

print("\nGenerated figure files:")
for f in sorted(os.listdir(FIG_DIR)):
    print(" -", f)

print("\nGenerated CSV files:")
for f in sorted(os.listdir(CSV_DIR)):
    print(" -", f)

Figures saved in: /content/fixed_cnn_inverse_project/figures
CSV files saved in: /content/fixed_cnn_inverse_project/csv

Generated figure files:
 - alpha_success_rate_iou_07.png
 - alpha_success_rate_relred_07.png
 - alpha_sweep_iou.png
 - alpha_sweep_iou_mean_std.png
 - alpha_sweep_loss_mean_std.png
 - alpha_sweep_mean_final_loss.png
 - alpha_sweep_rel_loss_reduction.png
 - alpha_sweep_relred_mean_std.png
 - convergence_bands_grad.png
 - convergence_bands_loss.png
 - curvature_alpha_mean.png
 - curvature_alpha_negative_fraction.png
 - fig_gradient_sparsity_per_optimizer.pdf
 - fig_gradient_sparsity_per_optimizer.png
 - grad_activity_active_fraction_main.png
 - grad_activity_frac_active_sp_gt_0.0001.png
 - grad_activity_frac_active_sp_gt_0.001.png
 - grad_activity_frac_active_sp_gt_0.01.png
 - grad_activity_frac_active_sp_gt_0.05.png
 - grad_activity_iou_vs_alpha.png
 - grad_activity_sp_mean.png
 - gradient_activity_spatial_examples.png
 - init_sweep_iou.png
 - init_sweep_mean_final_lo

In [ ]:
# Colab cell 27

def gradient_activity_map(problem: FixedCNNInverseProblem, x: np.ndarray):
    ax = problem.conv(x)
    s = sigmoid(ax, alpha=problem.alpha, c=problem.c)
    sp = sigmoid_prime_from_output(s, alpha=problem.alpha)
    return ax, s, sp

def gradient_activity_metrics(problem: FixedCNNInverseProblem,
                              x: np.ndarray,
                              eps_list=(1e-4, 1e-3, 1e-2, 5e-2)):
    ax, s, sp = gradient_activity_map(problem, x)

    metrics = {
        "sp_mean": float(np.mean(sp)),
        "sp_std": float(np.std(sp)),
        "sp_max": float(np.max(sp)),
        "sp_min": float(np.min(sp)),
        "ax_mean": float(np.mean(ax)),
        "ax_std": float(np.std(ax)),
        "sigmoid_mean": float(np.mean(s)),
        "sigmoid_std": float(np.std(s)),
    }

    for eps in eps_list:
        frac = np.mean(np.abs(sp) > eps)
        metrics[f"frac_active_sp_gt_{eps}"] = float(frac)

    return metrics, ax, s, sp

In [ ]:
import inspect
print(inspect.signature(OPTIMIZER_FUNCS["lbfgsb"]))

(problem: __main__.FixedCNNInverseProblem, x0: numpy.ndarray, maxiter: int = 200) -> Dict


In [ ]:
# Colab cell 28

def run_gradient_sparsity_alpha_sweep(
    image_shape=(64, 64),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alphas=(1.0, 2.0, 5.0, 10.0, 20.0, 40.0),
    c=0.5,
    optimizer_name="adam",
    init_mode="random",
    seeds=(0, 1, 2, 3, 4),
    optimizer_kwargs=None,
    use_final_x=True,
):
    if optimizer_kwargs is None:
        optimizer_kwargs = {"lr": 0.03, "steps": 200}

    rows = []
    saved_examples = []

    for alpha in alphas:
        for seed in seeds:
            payload = run_single_experiment(
                image_shape=image_shape,
                kernel_name=kernel_name,
                target_name=target_name,
                alpha=alpha,
                c=c,
                optimizer_name=optimizer_name,
                init_mode=init_mode,
                seed=seed,
                optimizer_kwargs=optimizer_kwargs
            )

            problem = payload["problem"]
            x_ref = payload["x_final"] if use_final_x else payload["x0"]

            gmetrics, ax, s, sp = gradient_activity_metrics(problem, x_ref)

            row = {
                "alpha": alpha,
                "seed": seed,
                "kernel_name": kernel_name,
                "target_name": target_name,
                "optimizer": optimizer_name,
                "use_final_x": use_final_x,
                **gmetrics,
                "loss_final": payload["summary"]["loss_final"],
                "output_iou_final": payload["summary"]["output_iou_final"],
                "rel_loss_reduction": payload["summary"]["rel_loss_reduction"],
            }
            rows.append(row)

            if seed == seeds[0]:
                saved_examples.append({
                    "alpha": alpha,
                    "payload": payload,
                    "ax": ax,
                    "sigmoid": s,
                    "sp": sp,
                })

            print(f"  [{optimizer_name}] alpha={alpha}, seed={seed} done")

    df = pd.DataFrame(rows)
    return df, saved_examples


# ── Run sweep for ALL optimizers ──────────────────────────────────────────────

OPTIMIZER_CONFIGS = {
    "adam":     {"lr": 0.03, "steps": 200},
    "pgd":      {"lr": 0.1,  "steps": 200},
    "momentum": {"lr": 0.1,  "steps": 200},
    "nesterov": {"lr": 0.1,  "steps": 200},
    "lbfgsb":   {"maxiter": 200},
}

ALPHAS_GRAD  = (1.0, 2.0, 5.0, 10.0, 20.0, 40.0)
SEEDS_GRAD   = (0, 1, 2, 3, 4)
KERNEL_GRAD  = "sobel_x"
TARGET_GRAD  = "checkerboard"
C_GRAD       = 0.5

all_dfs             = []
grad_sparse_examples = {}

for opt_name, opt_kwargs in OPTIMIZER_CONFIGS.items():
    print(f"\n{'='*50}")
    print(f"Running gradient sparsity sweep: {opt_name}")
    print(f"{'='*50}")

    df_opt, examples_opt = run_gradient_sparsity_alpha_sweep(
        kernel_name=KERNEL_GRAD,
        target_name=TARGET_GRAD,
        alphas=ALPHAS_GRAD,
        c=C_GRAD,
        optimizer_name=opt_name,
        seeds=SEEDS_GRAD,
        optimizer_kwargs=opt_kwargs,
        use_final_x=True,
    )

    all_dfs.append(df_opt)
    grad_sparse_examples[opt_name] = examples_opt
    print(f"  → {len(df_opt)} rows collected for {opt_name}")

# Combine and save
grad_sparse_df = pd.concat(all_dfs, ignore_index=True)
grad_sparse_df.to_csv(os.path.join(CSV_DIR, "grad_sparsity_all_optimizers.csv"), index=False)

# Sanity check
print(f"\nFinal shape: {grad_sparse_df.shape}")
print(f"Optimizers:  {grad_sparse_df['optimizer'].unique().tolist()}")
print("\nMean active fraction (|h'(Ax)| > 0.01) per optimizer per alpha:")
print(grad_sparse_df.groupby(["optimizer", "alpha"])["frac_active_sp_gt_0.01"].mean().unstack().round(3))

grad_sparse_df.head()


Running gradient sparsity sweep: adam
  [adam] alpha=1.0, seed=0 done
  [adam] alpha=1.0, seed=1 done
  [adam] alpha=1.0, seed=2 done
  [adam] alpha=1.0, seed=3 done
  [adam] alpha=1.0, seed=4 done
  [adam] alpha=2.0, seed=0 done
  [adam] alpha=2.0, seed=1 done
  [adam] alpha=2.0, seed=2 done
  [adam] alpha=2.0, seed=3 done
  [adam] alpha=2.0, seed=4 done
  [adam] alpha=5.0, seed=0 done
  [adam] alpha=5.0, seed=1 done
  [adam] alpha=5.0, seed=2 done
  [adam] alpha=5.0, seed=3 done
  [adam] alpha=5.0, seed=4 done
  [adam] alpha=10.0, seed=0 done
  [adam] alpha=10.0, seed=1 done
  [adam] alpha=10.0, seed=2 done
  [adam] alpha=10.0, seed=3 done
  [adam] alpha=10.0, seed=4 done
  [adam] alpha=20.0, seed=0 done
  [adam] alpha=20.0, seed=1 done
  [adam] alpha=20.0, seed=2 done
  [adam] alpha=20.0, seed=3 done
  [adam] alpha=20.0, seed=4 done
  [adam] alpha=40.0, seed=0 done
  [adam] alpha=40.0, seed=1 done
  [adam] alpha=40.0, seed=2 done
  [adam] alpha=40.0, seed=3 done
  [adam] alpha=40.0

/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


  [lbfgsb] alpha=1.0, seed=1 done
  [lbfgsb] alpha=1.0, seed=2 done
  [lbfgsb] alpha=1.0, seed=3 done
  [lbfgsb] alpha=1.0, seed=4 done
  [lbfgsb] alpha=2.0, seed=0 done
  [lbfgsb] alpha=2.0, seed=1 done
  [lbfgsb] alpha=2.0, seed=2 done
  [lbfgsb] alpha=2.0, seed=3 done
  [lbfgsb] alpha=2.0, seed=4 done
  [lbfgsb] alpha=5.0, seed=0 done
  [lbfgsb] alpha=5.0, seed=1 done
  [lbfgsb] alpha=5.0, seed=2 done
  [lbfgsb] alpha=5.0, seed=3 done
  [lbfgsb] alpha=5.0, seed=4 done
  [lbfgsb] alpha=10.0, seed=0 done
  [lbfgsb] alpha=10.0, seed=1 done
  [lbfgsb] alpha=10.0, seed=2 done
  [lbfgsb] alpha=10.0, seed=3 done
  [lbfgsb] alpha=10.0, seed=4 done
  [lbfgsb] alpha=20.0, seed=0 done
  [lbfgsb] alpha=20.0, seed=1 done
  [lbfgsb] alpha=20.0, seed=2 done
  [lbfgsb] alpha=20.0, seed=3 done
  [lbfgsb] alpha=20.0, seed=4 done
  [lbfgsb] alpha=40.0, seed=0 done
  [lbfgsb] alpha=40.0, seed=1 done
  [lbfgsb] alpha=40.0, seed=2 done
  [lbfgsb] alpha=40.0, seed=3 done
  [lbfgsb] alpha=40.0, seed=4 done

,alpha,seed,kernel_name,target_name,optimizer,use_final_x,sp_mean,sp_std,sp_max,sp_min,...,ax_std,sigmoid_mean,sigmoid_std,frac_active_sp_gt_0.0001,frac_active_sp_gt_0.001,frac_active_sp_gt_0.01,frac_active_sp_gt_0.05,loss_final,output_iou_final,rel_loss_reduction
0,1.0,0,sobel_x,checkerboard,adam,True,0.199459,0.045501,0.25,0.010866,...,0.974911,0.395262,0.198924,1.0,1.0,1.0,0.987061,516.633316,0.803711,0.579206
1,1.0,1,sobel_x,checkerboard,adam,True,0.198447,0.045277,0.25,0.010866,...,0.977718,0.394151,0.200871,1.0,1.0,1.0,0.988037,509.830958,0.811035,0.595989
2,1.0,2,sobel_x,checkerboard,adam,True,0.202783,0.042429,0.25,0.010866,...,0.895293,0.399000,0.192396,1.0,1.0,1.0,0.995117,490.551285,0.848633,0.591559
3,1.0,3,sobel_x,checkerboard,adam,True,0.202309,0.043243,0.25,0.010866,...,0.915136,0.397797,0.192991,1.0,1.0,1.0,0.993164,506.967046,0.813965,0.584898
4,1.0,4,sobel_x,checkerboard,adam,True,0.203355,0.042053,0.25,0.010866,...,0.886569,0.400011,0.191434,1.0,1.0,1.0,0.996338,500.374304,0.815918,0.598658


In [ ]:
print(grad_sparse_df['optimizer'].unique())
print(grad_sparse_df.shape)
all_dfs             = []
grad_sparse_examples = {}

for opt_name, opt_kwargs in OPTIMIZER_CONFIGS.items():
    print(f"Starting: {opt_name}")
    df_opt, examples_opt = run_gradient_sparsity_alpha_sweep(
        kernel_name=KERNEL_GRAD,
        target_name=TARGET_GRAD,
        alphas=ALPHAS_GRAD,
        c=C_GRAD,
        optimizer_name=opt_name,
        seeds=SEEDS_GRAD,
        optimizer_kwargs=opt_kwargs,
        use_final_x=True,
    )
    all_dfs.append(df_opt)
    grad_sparse_examples[opt_name] = examples_opt
    print(f"  ✓ {opt_name} done — {len(df_opt)} rows")

grad_sparse_df = pd.concat(all_dfs, ignore_index=True)
print(f"\nDone. Optimizers: {grad_sparse_df['optimizer'].unique().tolist()}")
print(f"Shape: {grad_sparse_df.shape}")

['adam' 'pgd' 'momentum' 'nesterov' 'lbfgsb']
(150, 21)
Starting: adam
  [adam] alpha=1.0, seed=0 done
  [adam] alpha=1.0, seed=1 done
  [adam] alpha=1.0, seed=2 done
  [adam] alpha=1.0, seed=3 done
  [adam] alpha=1.0, seed=4 done
  [adam] alpha=2.0, seed=0 done
  [adam] alpha=2.0, seed=1 done
  [adam] alpha=2.0, seed=2 done
  [adam] alpha=2.0, seed=3 done
  [adam] alpha=2.0, seed=4 done
  [adam] alpha=5.0, seed=0 done
  [adam] alpha=5.0, seed=1 done
  [adam] alpha=5.0, seed=2 done
  [adam] alpha=5.0, seed=3 done
  [adam] alpha=5.0, seed=4 done
  [adam] alpha=10.0, seed=0 done
  [adam] alpha=10.0, seed=1 done
  [adam] alpha=10.0, seed=2 done
  [adam] alpha=10.0, seed=3 done
  [adam] alpha=10.0, seed=4 done
  [adam] alpha=20.0, seed=0 done
  [adam] alpha=20.0, seed=1 done
  [adam] alpha=20.0, seed=2 done
  [adam] alpha=20.0, seed=3 done
  [adam] alpha=20.0, seed=4 done
  [adam] alpha=40.0, seed=0 done
  [adam] alpha=40.0, seed=1 done
  [adam] alpha=40.0, seed=2 done
  [adam] alpha=40.0,

/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


  [lbfgsb] alpha=2.0, seed=0 done
  [lbfgsb] alpha=2.0, seed=1 done
  [lbfgsb] alpha=2.0, seed=2 done
  [lbfgsb] alpha=2.0, seed=3 done
  [lbfgsb] alpha=2.0, seed=4 done
  [lbfgsb] alpha=5.0, seed=0 done
  [lbfgsb] alpha=5.0, seed=1 done
  [lbfgsb] alpha=5.0, seed=2 done
  [lbfgsb] alpha=5.0, seed=3 done
  [lbfgsb] alpha=5.0, seed=4 done
  [lbfgsb] alpha=10.0, seed=0 done
  [lbfgsb] alpha=10.0, seed=1 done
  [lbfgsb] alpha=10.0, seed=2 done
  [lbfgsb] alpha=10.0, seed=3 done
  [lbfgsb] alpha=10.0, seed=4 done
  [lbfgsb] alpha=20.0, seed=0 done
  [lbfgsb] alpha=20.0, seed=1 done
  [lbfgsb] alpha=20.0, seed=2 done
  [lbfgsb] alpha=20.0, seed=3 done
  [lbfgsb] alpha=20.0, seed=4 done
  [lbfgsb] alpha=40.0, seed=0 done
  [lbfgsb] alpha=40.0, seed=1 done
  [lbfgsb] alpha=40.0, seed=2 done
  [lbfgsb] alpha=40.0, seed=3 done
  [lbfgsb] alpha=40.0, seed=4 done
  ✓ lbfgsb done — 30 rows

Done. Optimizers: ['adam', 'pgd', 'momentum', 'nesterov', 'lbfgsb']
Shape: (150, 21)


In [ ]:
# Colab cell 29

def plot_gradient_sparsity_alpha_sweep(df: pd.DataFrame):
    metrics_to_plot = [
        "sp_mean",
        "frac_active_sp_gt_0.0001",
        "frac_active_sp_gt_0.001",
        "frac_active_sp_gt_0.01",
        "frac_active_sp_gt_0.05",
    ]

    for metric in metrics_to_plot:
        grp = df.groupby("alpha")[metric].agg(["mean", "std"]).reset_index()

        plt.figure(figsize=(6, 4))
        plt.errorbar(
            grp["alpha"],
            grp["mean"],
            yerr=grp["std"],
            marker="o",
            capsize=4
        )
        plt.xlabel("alpha")
        plt.ylabel(metric)
        plt.title(f"Gradient activity vs alpha: {metric}")
        save_fig(os.path.join(FIG_DIR, f"grad_activity_{metric}.png"))
        plt.show()

    # Original single-curve paper figure (kept for backwards compat)
    grp1 = df.groupby("alpha")["frac_active_sp_gt_0.01"].agg(["mean", "std"]).reset_index()
    plt.figure(figsize=(6, 4))
    plt.errorbar(grp1["alpha"], grp1["mean"], yerr=grp1["std"], marker="o", capsize=4)
    plt.xlabel("alpha")
    plt.ylabel("Fraction active: |h'(Ax)| > 0.01")
    plt.title("Active gradient fraction decreases with alpha")
    save_fig(os.path.join(FIG_DIR, "grad_activity_active_fraction_main.png"))
    plt.show()

    grp2 = df.groupby("alpha")["output_iou_final"].agg(["mean", "std"]).reset_index()
    plt.figure(figsize=(6, 4))
    plt.errorbar(grp2["alpha"], grp2["mean"], yerr=grp2["std"], marker="o", capsize=4)
    plt.xlabel("alpha")
    plt.ylabel("Final output IoU")
    plt.title("Reconstruction IoU vs alpha")
    save_fig(os.path.join(FIG_DIR, "grad_activity_iou_vs_alpha.png"))
    plt.show()


def plot_gradient_sparsity_per_optimizer(df: pd.DataFrame):
    """
    NEW: Per-optimizer active gradient fraction at convergence vs alpha.
    This is the paper figure for §6.4.
    """
    THRESHOLD = "frac_active_sp_gt_0.01"
    OPTIMIZERS = ["adam", "lbfgsb", "momentum", "nesterov", "pgd"]

    STYLES = {
        "adam":     {"color": "#1f77b4", "marker": "o"},
        "lbfgsb":   {"color": "#ff7f0e", "marker": "s"},
        "momentum": {"color": "#2ca02c", "marker": "^"},
        "nesterov": {"color": "#d62728", "marker": "D"},
        "pgd":      {"color": "#9467bd", "marker": "v"},
    }

    fig, ax = plt.subplots(figsize=(7, 4.5))

    for opt in OPTIMIZERS:
        sub = df[df["optimizer"] == opt]
        if sub.empty:
            print(f"  WARNING: no rows found for optimizer='{opt}', skipping.")
            continue

        grp = sub.groupby("alpha")[THRESHOLD].agg(["mean", "std"]).reset_index()
        style = STYLES.get(opt, {"color": "gray", "marker": "o"})

        ax.errorbar(
            grp["alpha"],
            grp["mean"],
            yerr=grp["std"],
            label=opt,
            marker=style["marker"],
            color=style["color"],
            capsize=4,
            linewidth=1.8,
            markersize=6,
        )

    ax.set_xlabel("alpha", fontsize=12)
    ax.set_ylabel("Fraction active: |h'(Ax)| > 0.01", fontsize=11)
    ax.set_title("Active gradient fraction at convergence (per optimizer)", fontsize=12)
    ax.legend(fontsize=10, framealpha=0.9)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, linestyle="--", alpha=0.4)

    plt.tight_layout()
    save_fig(os.path.join(FIG_DIR, "fig_gradient_sparsity_per_optimizer.png"))
    save_fig(os.path.join(FIG_DIR, "fig_gradient_sparsity_per_optimizer.png"))
    plt.show()
    print("Per-optimizer gradient sparsity figure saved.")

    # Summary table
    print("\nSummary: mean active fraction at convergence")
    print(f"{'alpha':>8}", end="")
    for opt in OPTIMIZERS:
        print(f"  {opt:>10}", end="")
    print()
    for alpha in sorted(df["alpha"].unique()):
        print(f"{alpha:>8.1f}", end="")
        for opt in OPTIMIZERS:
            sub = df[(df["optimizer"] == opt) & (df["alpha"] == alpha)]
            val = sub[THRESHOLD].mean() if not sub.empty else float("nan")
            print(f"  {val:>10.3f}", end="")
        print()


# Run both
plot_gradient_sparsity_alpha_sweep(grad_sparse_df)
plot_gradient_sparsity_per_optimizer(grad_sparse_df)
print("All gradient sparsity plots saved.")

Per-optimizer gradient sparsity figure saved.

Summary: mean active fraction at convergence
   alpha        adam      lbfgsb    momentum    nesterov         pgd
     1.0       1.000       1.000       1.000       1.000       1.000
     2.0       0.981       0.980       0.983       0.980       0.987
     5.0       0.654       0.633       0.438       0.413       0.557
    10.0       0.345       0.247       0.226       0.224       0.164
    20.0       0.146       0.068       0.024       0.031       0.051
    40.0       0.060       0.029       0.005       0.009       0.027
All gradient sparsity plots saved.


In [ ]:
# Colab cell 30

def plot_spatial_gradient_activity_examples(saved_examples, max_examples=4):
    # saved_examples can be a dict (keyed by optimizer) or a list — handle both
    if isinstance(saved_examples, dict):
        # Use adam by default for spatial visualization; fall back to first key
        key = "adam" if "adam" in saved_examples else list(saved_examples.keys())[0]
        saved_examples = saved_examples[key]

    n = min(len(saved_examples), max_examples)
    fig, axes = plt.subplots(n, 4, figsize=(12, 3 * n))

    if n == 1:
        axes = np.array([axes])

    for i in range(n):
        item = saved_examples[i]
        alpha = item["alpha"]
        payload = item["payload"]
        sp = item["sp"]
        ax_map = item["ax"]
        sig = item["sigmoid"]
        x_final = payload["x_final"]

        axes[i, 0].imshow(x_final, cmap="gray", vmin=0, vmax=1)
        axes[i, 0].set_title(f"x_final\nalpha={alpha}")

        axes[i, 1].imshow(ax_map, cmap="gray")
        axes[i, 1].set_title("Ax")

        axes[i, 2].imshow(sig, cmap="gray", vmin=0, vmax=1)
        axes[i, 2].set_title("h(Ax)")

        axes[i, 3].imshow(sp, cmap="magma")
        axes[i, 3].set_title("h'(Ax)")

        for j in range(4):
            axes[i, j].axis("off")

    plt.tight_layout()
    save_fig(os.path.join(FIG_DIR, "gradient_activity_spatial_examples.png"))

plot_spatial_gradient_activity_examples(grad_sparse_examples, max_examples=4)
print("Saved derivative-map figure.")

Saved derivative-map figure.


In [ ]:
# Colab cell 31

def plot_mean_std_lines(df: pd.DataFrame,
                        x_col: str,
                        y_col: str,
                        group_col: str,
                        title: str,
                        xlabel: str,
                        ylabel: str,
                        save_name: str,
                        sort_x=True):
    plt.figure(figsize=(7, 5))

    for group in sorted(df[group_col].unique()):
        sub = df[df[group_col] == group]
        grp = sub.groupby(x_col)[y_col].agg(["mean", "std"]).reset_index()
        if sort_x:
            grp = grp.sort_values(x_col)

        x = grp[x_col].values
        y = grp["mean"].values
        s = grp["std"].values

        plt.plot(x, y, marker="o", label=str(group))
        plt.fill_between(x, y - s, y + s, alpha=0.18)

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    save_fig(os.path.join(FIG_DIR, save_name))

In [ ]:
# Colab cell 32

plot_mean_std_lines(
    alpha_df,
    x_col="alpha",
    y_col="loss_final",
    group_col="optimizer",
    title="Effect of sigmoid stiffness on final loss",
    xlabel="alpha",
    ylabel="Final loss",
    save_name="alpha_sweep_loss_mean_std.png"
)

plot_mean_std_lines(
    alpha_df,
    x_col="alpha",
    y_col="output_iou_final",
    group_col="optimizer",
    title="Effect of sigmoid stiffness on output IoU",
    xlabel="alpha",
    ylabel="Output IoU",
    save_name="alpha_sweep_iou_mean_std.png"
)

plot_mean_std_lines(
    alpha_df,
    x_col="alpha",
    y_col="rel_loss_reduction",
    group_col="optimizer",
    title="Effect of sigmoid stiffness on relative loss reduction",
    xlabel="alpha",
    ylabel="Relative loss reduction",
    save_name="alpha_sweep_relred_mean_std.png"
)

print("Saved alpha mean±std plots.")

Saved alpha mean±std plots.


In [ ]:
# Colab cell 33

def plot_categorical_mean_std(df: pd.DataFrame,
                              category_col: str,
                              y_col: str,
                              group_col: str,
                              title: str,
                              ylabel: str,
                              save_name: str):
    plt.figure(figsize=(9, 5))
    categories = list(sorted(df[category_col].unique()))
    x = np.arange(len(categories))

    for group in sorted(df[group_col].unique()):
        sub = df[df[group_col] == group]
        grp = sub.groupby(category_col)[y_col].agg(["mean", "std"]).reindex(categories).reset_index()

        y = grp["mean"].values
        s = grp["std"].values
        plt.plot(x, y, marker="o", label=str(group))
        plt.fill_between(x, y - s, y + s, alpha=0.18)

    plt.xticks(x, categories, rotation=20)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    save_fig(os.path.join(FIG_DIR, save_name))

plot_categorical_mean_std(
    kernel_df,
    category_col="kernel_name",
    y_col="loss_final",
    group_col="optimizer",
    title="Final loss across convolution kernels",
    ylabel="Final loss",
    save_name="kernel_sweep_loss_mean_std.png"
)

plot_categorical_mean_std(
    kernel_df,
    category_col="kernel_name",
    y_col="output_iou_final",
    group_col="optimizer",
    title="Output IoU across convolution kernels",
    ylabel="Output IoU",
    save_name="kernel_sweep_iou_mean_std.png"
)

print("Saved kernel mean±std plots.")

Saved kernel mean±std plots.


In [ ]:
# Colab cell 34

def add_success_columns(df: pd.DataFrame,
                        iou_thresholds=(0.6, 0.7, 0.8),
                        relred_thresholds=(0.5, 0.7)):
    out = df.copy()

    for thr in iou_thresholds:
        out[f"success_iou_ge_{thr}"] = (out["output_iou_final"] >= thr).astype(int)

    for thr in relred_thresholds:
        out[f"success_relred_ge_{thr}"] = (out["rel_loss_reduction"] >= thr).astype(int)

    return out

alpha_df_success = add_success_columns(alpha_df)
alpha_df_success.head()

,image_h,image_w,kernel_name,target_name,alpha,c,optimizer,init_mode,seed,time_sec,...,rel_loss_reduction,num_iters_recorded,scipy_message,scipy_success,scipy_nit,success_iou_ge_0.6,success_iou_ge_0.7,success_iou_ge_0.8,success_relred_ge_0.5,success_relred_ge_0.7
0,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,0,0.053199,...,0.585968,200,NaN,NaN,NaN,1,1,0,1,0
1,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,1,0.052152,...,0.591705,200,NaN,NaN,NaN,1,1,1,1,0
2,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,2,0.052189,...,0.604143,200,NaN,NaN,NaN,1,1,1,1,0
3,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,3,0.052403,...,0.591987,200,NaN,NaN,NaN,1,1,1,1,0
4,64,64,sobel_x,checkerboard,1.0,0.5,pgd,random,4,0.052381,...,0.605142,200,NaN,NaN,NaN,1,1,1,1,0


In [ ]:
# Colab cell 35

def plot_success_rate_vs_alpha(df: pd.DataFrame, success_col: str, save_name: str, title: str):
    plt.figure(figsize=(7, 5))
    for opt in sorted(df["optimizer"].unique()):
        sub = df[df["optimizer"] == opt]
        grp = sub.groupby("alpha")[success_col].mean().reset_index().sort_values("alpha")
        plt.plot(grp["alpha"], grp[success_col], marker="o", label=opt)

    plt.xlabel("alpha")
    plt.ylabel("Success rate")
    plt.title(title)
    plt.ylim(-0.02, 1.02)
    plt.legend()
    save_fig(os.path.join(FIG_DIR, save_name))

plot_success_rate_vs_alpha(
    alpha_df_success,
    success_col="success_iou_ge_0.7",
    save_name="alpha_success_rate_iou_07.png",
    title="Success rate vs alpha (IoU ≥ 0.7)"
)

plot_success_rate_vs_alpha(
    alpha_df_success,
    success_col="success_relred_ge_0.7",
    save_name="alpha_success_rate_relred_07.png",
    title="Success rate vs alpha (relative loss reduction ≥ 0.7)"
)

print("Saved success-rate plots.")

Saved success-rate plots.


In [ ]:
# Colab cell 36

def run_convergence_band_experiment(
    image_shape=(64, 64),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alpha=10.0,
    c=0.5,
    optimizers=None,
    init_mode="random",
    seeds=(0, 1, 2, 3, 4)
):
    if optimizers is None:
        optimizers = {
            "pgd": {"lr": 0.10, "steps": 200},
            "momentum": {"lr": 0.05, "beta": 0.9, "steps": 200},
            "adam": {"lr": 0.03, "steps": 200},
            "lbfgsb": {"maxiter": 200},
        }

    results = {}

    for opt_name, opt_kwargs in optimizers.items():
        loss_curves = []
        grad_curves = []

        for seed in seeds:
            payload = run_single_experiment(
                image_shape=image_shape,
                kernel_name=kernel_name,
                target_name=target_name,
                alpha=alpha,
                c=c,
                optimizer_name=opt_name,
                init_mode=init_mode,
                seed=seed,
                optimizer_kwargs=opt_kwargs
            )

            loss_hist = np.array(payload["result"]["loss_hist"], dtype=np.float64)
            grad_hist = np.array(payload["result"]["grad_hist"], dtype=np.float64)

            loss_curves.append(loss_hist)
            grad_curves.append(grad_hist)

            print(f"done opt={opt_name}, seed={seed}")

        min_len = min(len(x) for x in loss_curves)
        loss_arr = np.stack([x[:min_len] for x in loss_curves], axis=0)
        grad_arr = np.stack([x[:min_len] for x in grad_curves], axis=0)

        results[opt_name] = {
            "loss_mean": loss_arr.mean(axis=0),
            "loss_std": loss_arr.std(axis=0),
            "grad_mean": grad_arr.mean(axis=0),
            "grad_std": grad_arr.std(axis=0),
        }

    return results

conv_band_results = run_convergence_band_experiment()

done opt=pgd, seed=0
done opt=pgd, seed=1
done opt=pgd, seed=2
done opt=pgd, seed=3
done opt=pgd, seed=4
done opt=momentum, seed=0
done opt=momentum, seed=1
done opt=momentum, seed=2
done opt=momentum, seed=3
done opt=momentum, seed=4
done opt=adam, seed=0
done opt=adam, seed=1
done opt=adam, seed=2
done opt=adam, seed=3
done opt=adam, seed=4


/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


done opt=lbfgsb, seed=0
done opt=lbfgsb, seed=1
done opt=lbfgsb, seed=2
done opt=lbfgsb, seed=3
done opt=lbfgsb, seed=4


In [ ]:
# Colab cell 37

def plot_convergence_bands(results: Dict[str, Dict[str, np.ndarray]],
                           quantity: str,
                           ylabel: str,
                           title: str,
                           save_name: str):
    plt.figure(figsize=(7, 5))

    for opt_name, stats in results.items():
        mean = stats[f"{quantity}_mean"]
        std = stats[f"{quantity}_std"]
        x = np.arange(len(mean))

        plt.plot(x, mean, label=opt_name)
        plt.fill_between(x, mean - std, mean + std, alpha=0.18)

    plt.xlabel("Iteration")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    save_fig(os.path.join(FIG_DIR, save_name))

plot_convergence_bands(
    conv_band_results,
    quantity="loss",
    ylabel="Loss",
    title="Convergence curves with mean ± std",
    save_name="convergence_bands_loss.png"
)

plot_convergence_bands(
    conv_band_results,
    quantity="grad",
    ylabel="Gradient norm",
    title="Gradient norm curves with mean ± std",
    save_name="convergence_bands_grad.png"
)

print("Saved convergence-band plots.")

Saved convergence-band plots.


In [ ]:
# Colab cell 38

def directional_second_difference(problem: FixedCNNInverseProblem,
                                  x: np.ndarray,
                                  v: np.ndarray,
                                  eps: float = 1e-3):
    v = v / (np.linalg.norm(v) + 1e-12)
    x_flat = x.reshape(-1)
    shape = x.shape

    xp = project_box((x_flat + eps * v).reshape(shape))
    xm = project_box((x_flat - eps * v).reshape(shape))
    x0 = project_box(x.copy())

    return (problem.loss(xp) - 2.0 * problem.loss(x0) + problem.loss(xm)) / (eps ** 2)

def sample_directional_curvatures(problem: FixedCNNInverseProblem,
                                  x: np.ndarray,
                                  num_dirs: int = 50,
                                  eps: float = 1e-3,
                                  seed: int = 0):
    rng = np.random.default_rng(seed)
    n = x.size
    vals = []

    for _ in range(num_dirs):
        v = rng.normal(size=n)
        curv = directional_second_difference(problem, x, v, eps=eps)
        vals.append(curv)

    vals = np.array(vals, dtype=np.float64)
    return {
        "curv_mean": float(np.mean(vals)),
        "curv_std": float(np.std(vals)),
        "curv_min": float(np.min(vals)),
        "curv_max": float(np.max(vals)),
        "curv_frac_negative": float(np.mean(vals < 0)),
        "curv_values": vals,
    }

In [ ]:
# Colab cell 39

def run_curvature_alpha_sweep(
    image_shape=(32, 32),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alphas=(1.0, 2.0, 5.0, 10.0, 20.0, 40.0),
    c=0.5,
    optimizer_name="adam",
    seeds=(0, 1, 2),
    optimizer_kwargs=None
):
    if optimizer_kwargs is None:
        optimizer_kwargs = {"lr": 0.03, "steps": 120}

    rows = []

    for alpha in alphas:
        for seed in seeds:
            payload = run_single_experiment(
                image_shape=image_shape,
                kernel_name=kernel_name,
                target_name=target_name,
                alpha=alpha,
                c=c,
                optimizer_name=optimizer_name,
                init_mode="random",
                seed=seed,
                optimizer_kwargs=optimizer_kwargs
            )

            curv = sample_directional_curvatures(
                payload["problem"],
                payload["x_final"],
                num_dirs=40,
                eps=1e-3,
                seed=seed
            )

            rows.append({
                "alpha": alpha,
                "seed": seed,
                "curv_mean": curv["curv_mean"],
                "curv_std": curv["curv_std"],
                "curv_min": curv["curv_min"],
                "curv_max": curv["curv_max"],
                "curv_frac_negative": curv["curv_frac_negative"],
            })
            print(f"done alpha={alpha}, seed={seed}")

    df = pd.DataFrame(rows)
    path = os.path.join(CSV_DIR, "curvature_alpha_sweep.csv")
    df.to_csv(path, index=False)
    print("Saved:", path)
    return df

curv_alpha_df = run_curvature_alpha_sweep()
curv_alpha_df.head()

done alpha=1.0, seed=0
done alpha=1.0, seed=1
done alpha=1.0, seed=2
done alpha=2.0, seed=0
done alpha=2.0, seed=1
done alpha=2.0, seed=2
done alpha=5.0, seed=0
done alpha=5.0, seed=1
done alpha=5.0, seed=2
done alpha=10.0, seed=0
done alpha=10.0, seed=1
done alpha=10.0, seed=2
done alpha=20.0, seed=0
done alpha=20.0, seed=1
done alpha=20.0, seed=2
done alpha=40.0, seed=0
done alpha=40.0, seed=1
done alpha=40.0, seed=2
Saved: /content/fixed_cnn_inverse_project/csv/curvature_alpha_sweep.csv


,alpha,seed,curv_mean,curv_std,curv_min,curv_max,curv_frac_negative
0,1.0,0,4505.755660,192.994823,4052.269939,4938.573294,0.0
1,1.0,1,4233.442341,295.720802,3701.935066,4953.771180,0.0
2,1.0,2,4215.393189,259.623456,3722.906604,4725.149827,0.0
3,2.0,0,5002.577897,247.599943,4447.826233,5410.227244,0.0
4,2.0,1,4911.194456,334.460440,4393.900323,5669.693954,0.0


In [ ]:
# Colab cell 40

plot_mean_std_lines(
    curv_alpha_df,
    x_col="alpha",
    y_col="curv_mean",
    group_col="seed",
    title="Directional curvature mean vs alpha",
    xlabel="alpha",
    ylabel="Mean directional curvature",
    save_name="curvature_alpha_mean.png"
)

# Better aggregate-only version
grp = curv_alpha_df.groupby("alpha")["curv_frac_negative"].agg(["mean", "std"]).reset_index()
plt.figure(figsize=(6, 4))
plt.errorbar(grp["alpha"], grp["mean"], yerr=grp["std"], marker="o", capsize=4)
plt.xlabel("alpha")
plt.ylabel("Fraction of negative curvature directions")
plt.title("Negative-curvature frequency vs alpha")
save_fig(os.path.join(FIG_DIR, "curvature_alpha_negative_fraction.png"))

print("Saved curvature plots.")

Saved curvature plots.


In [ ]:
# Colab cell 41

def make_mean_std_table(df: pd.DataFrame,
                        group_cols,
                        metric_cols,
                        round_digits=4):
    rows = []
    grouped = df.groupby(group_cols)

    for key, sub in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        row = {col: val for col, val in zip(group_cols, key)}

        for m in metric_cols:
            row[f"{m}_mean"] = round(float(sub[m].mean()), round_digits)
            row[f"{m}_std"] = round(float(sub[m].std()), round_digits)

        rows.append(row)

    return pd.DataFrame(rows)

alpha_table_polished = make_mean_std_table(
    alpha_df,
    group_cols=["alpha", "optimizer"],
    metric_cols=["loss_final", "output_iou_final", "rel_loss_reduction", "time_sec"]
)
alpha_table_polished.to_csv(os.path.join(CSV_DIR, "alpha_table_polished.csv"), index=False)

kernel_table_polished = make_mean_std_table(
    kernel_df,
    group_cols=["kernel_name", "optimizer"],
    metric_cols=["loss_final", "output_iou_final", "rel_loss_reduction", "time_sec"]
)
kernel_table_polished.to_csv(os.path.join(CSV_DIR, "kernel_table_polished.csv"), index=False)

grad_sparse_table = make_mean_std_table(
    grad_sparse_df,
    group_cols=["alpha"],
    metric_cols=["sp_mean", "frac_active_sp_gt_0.01", "output_iou_final", "rel_loss_reduction"]
)
grad_sparse_table.to_csv(os.path.join(CSV_DIR, "grad_sparse_table.csv"), index=False)

print("Saved polished tables.")
alpha_table_polished.head(12)

Saved polished tables.


,alpha,optimizer,loss_final_mean,loss_final_std,output_iou_final_mean,output_iou_final_std,rel_loss_reduction_mean,rel_loss_reduction_std,time_sec_mean,time_sec_std
0,1.0,adam,504.8714,9.9073,0.8187,0.0174,0.5901,0.0080,0.0558,0.0003
1,1.0,lbfgsb,493.7918,11.4852,0.8335,0.0086,0.5991,0.0093,0.0376,0.0077
2,1.0,momentum,498.7809,15.2472,0.8300,0.0116,0.5950,0.0130,0.0527,0.0002
3,1.0,nesterov,499.2109,13.6138,0.8216,0.0098,0.5947,0.0093,0.0541,0.0001
4,1.0,pgd,497.9209,15.3751,0.8161,0.0270,0.5958,0.0084,0.0525,0.0004
5,2.0,adam,337.2418,13.4623,0.8720,0.0140,0.7711,0.0067,0.0555,0.0001
6,2.0,lbfgsb,336.9504,12.0487,0.8694,0.0052,0.7713,0.0054,0.0811,0.0192
7,2.0,momentum,328.4605,7.8755,0.8701,0.0067,0.7770,0.0047,0.0524,0.0001
8,2.0,nesterov,322.7825,10.0876,0.8698,0.0074,0.7809,0.0049,0.0539,0.0001
9,2.0,pgd,431.1533,7.4383,0.6793,0.0044,0.7073,0.0067,0.0521,0.0002


In [ ]:
# New Experiment: Threshold Sensitivity Sweep
def run_threshold_sweep(
    image_shape=(64, 64),
    kernel_name="sobel_x",
    target_name="checkerboard",
    alpha=10.0,
    thresholds=(0.1, 0.3, 0.5, 0.7, 0.9),
    optimizers=("pgd", "adam", "lbfgsb"),
    seeds=(0, 1, 2)
):
    all_summaries = []

    for c in thresholds:
        print(f"Running sweep for c = {c}")
        for opt in optimizers:
            # Set standard params for each optimizer
            opt_kwargs = {}
            if opt == "pgd": opt_kwargs = {"lr": 0.1, "steps": 200}
            if opt == "adam": opt_kwargs = {"lr": 0.03, "steps": 200}
            if opt == "lbfgsb": opt_kwargs = {"maxiter": 200}

            for seed in seeds:
                payload = run_single_experiment(
                    image_shape=image_shape,
                    kernel_name=kernel_name,
                    target_name=target_name,
                    alpha=alpha,
                    c=c,
                    optimizer_name=opt,
                    seed=seed,
                    optimizer_kwargs=opt_kwargs
                )
                all_summaries.append(payload["summary"])

    return pd.DataFrame(all_summaries)

# Execute the sweep
threshold_df = run_threshold_sweep()

# Plotting the results
def plot_threshold_impact(df: pd.DataFrame):
    plt.figure(figsize=(10, 5))

    # Plot 1: Final Loss vs Threshold
    plt.subplot(1, 2, 1)
    for opt in df["optimizer"].unique():
        sub = df[df["optimizer"] == opt]
        grp = sub.groupby("c")["loss_final"].mean()
        plt.plot(grp.index, grp.values, marker='o', label=opt)
    plt.xlabel("Threshold (c)")
    plt.ylabel("Mean Final Loss")
    plt.title("Impact of c on Final Loss")
    plt.legend()

    # Plot 2: Success Rate (IoU > 0.7) vs Threshold
    plt.subplot(1, 2, 2)
    for opt in df["optimizer"].unique():
        sub = df[df["optimizer"] == opt]
        # Calculate fraction of runs where IoU > 0.7
        success = sub.groupby("c")["output_iou_final"].apply(lambda x: (x > 0.7).mean())
        plt.plot(success.index, success.values, marker='s', label=opt)
    plt.xlabel("Threshold (c)")
    plt.ylabel("Success Rate (IoU > 0.7)")
    plt.title("Impact of c on Success Rate")
    plt.legend()

    save_fig(os.path.join(FIG_DIR, "threshold_sensitivity_analysis.png"))
    plt.show()

plot_threshold_impact(threshold_df)



Running sweep for c = 0.1


/tmp/ipykernel_17592/2011133143.py:136: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


Running sweep for c = 0.3
Running sweep for c = 0.5
Running sweep for c = 0.7
Running sweep for c = 0.9


In [ ]:
def plot_2d_landscape_slice(problem, x_center, n_grid=60, radius=0.8, seed=0, save_name="landscape_2d_slice.png"):
    rng = np.random.default_rng(seed)
    n = x_center.size
    # Two random orthogonal directions
    u = rng.normal(size=n); u /= np.linalg.norm(u)
    v = rng.normal(size=n); v -= v.dot(u) * u; v /= np.linalg.norm(v)

    ts = np.linspace(-radius, radius, n_grid)
    Z = np.zeros((n_grid, n_grid))
    for i, ti in enumerate(ts):
        for j, tj in enumerate(ts):
            xij = project_box((x_center.reshape(-1) + ti * u + tj * v).reshape(problem.image_shape))
            Z[i, j] = problem.loss(xij)

    plt.figure(figsize=(5, 4))
    plt.contourf(ts, ts, Z, levels=30, cmap="viridis")
    plt.colorbar(label="Loss J(x)")
    plt.xlabel("Direction u coefficient"); plt.ylabel("Direction v coefficient")
    plt.title(f"Landscape slice near solution (alpha={problem.alpha})")
    save_fig(os.path.join(FIG_DIR, save_name))
    return Z

# Run for alpha=2 and alpha=20 for side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, alpha_val in zip(axes, [2.0, 20.0]):
    payload = run_single_experiment(kernel_name="sobel_x", target_name="checkerboard",
                                    alpha=alpha_val, c=0.5, optimizer_name="adam",
                                    optimizer_kwargs={"lr": 0.03, "steps": 300})
    x_sol = payload["x_final"].reshape(-1)
    problem = payload["problem"]
    rng = np.random.default_rng(0)
    n = x_sol.size
    u = rng.normal(size=n); u /= np.linalg.norm(u)
    v = rng.normal(size=n); v -= v.dot(u)*u; v /= np.linalg.norm(v)
    ts = np.linspace(-0.8, 0.8, 50)
    Z = np.zeros((50, 50))
    for i, ti in enumerate(ts):
        for j, tj in enumerate(ts):
            xij = project_box((x_sol + ti*u + tj*v).reshape(problem.image_shape))
            Z[i, j] = problem.loss(xij)
    im = ax.contourf(ts, ts, Z, levels=30, cmap="viridis")
    plt.colorbar(im, ax=ax)
    ax.set_title(f"alpha={alpha_val}")
    ax.set_xlabel("Direction u"); ax.set_ylabel("Direction v")

plt.suptitle("2D landscape slices near solution")
plt.tight_layout()
save_fig(os.path.join(FIG_DIR, "landscape_2d_comparison.pdf"))

In [ ]:
def run_nonuniqueness_experiment(kernel_name="sobel_x", target_name="checkerboard",
                                  alpha=20.0, c=0.5, n_restarts=20):
    solutions = []
    for seed in range(n_restarts):
        payload = run_single_experiment(kernel_name=kernel_name, target_name=target_name,
                                         alpha=alpha, c=0.5, optimizer_name="adam",
                                         seed=seed, optimizer_kwargs={"lr": 0.03, "steps": 300})
        solutions.append(payload["x_final"].reshape(-1))
        print(f"restart {seed}: loss={payload['summary']['loss_final']:.3f}, iou={payload['summary']['output_iou_final']:.3f}")

    solutions = np.stack(solutions)  # (n_restarts, N)
    # Pairwise L2 distances
    dists = []
    for i in range(n_restarts):
        for j in range(i+1, n_restarts):
            dists.append(np.linalg.norm(solutions[i] - solutions[j]))
    dists = np.array(dists)

    plt.figure(figsize=(6, 4))
    plt.hist(dists, bins=30, edgecolor="white")
    plt.xlabel("Pairwise L2 distance between x_final solutions")
    plt.ylabel("Count")
    plt.title(f"Solution diversity across {n_restarts} random restarts (alpha={alpha})")
    save_fig(os.path.join(FIG_DIR, "nonuniqueness_pairwise_dist.pdf"))

    # Mosaic of 6 diverse solutions
    fig, axes = plt.subplots(2, 3, figsize=(9, 6))
    for idx, ax in enumerate(axes.ravel()):
        ax.imshow(solutions[idx*3].reshape(64, 64), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"Restart {idx*3}")
        ax.axis("off")
    plt.suptitle(f"Diverse solutions found (alpha={alpha}, same target)")
    save_fig(os.path.join(FIG_DIR, "nonuniqueness_mosaic.pdf"))

    return solutions, dists

solutions, dists = run_nonuniqueness_experiment()
print(f"Mean pairwise distance: {dists.mean():.3f} ± {dists.std():.3f}")
print(f"Min: {dists.min():.3f}, Max: {dists.max():.3f}")

restart 0: loss=644.260, iou=0.691
restart 1: loss=622.197, iou=0.702
restart 2: loss=637.195, iou=0.694
restart 3: loss=620.961, iou=0.701
restart 4: loss=626.659, iou=0.702
restart 5: loss=647.778, iou=0.690
restart 6: loss=662.106, iou=0.683
restart 7: loss=642.187, iou=0.691
restart 8: loss=651.125, iou=0.688
restart 9: loss=664.386, iou=0.685
restart 10: loss=619.263, iou=0.704
restart 11: loss=619.990, iou=0.704
restart 12: loss=637.245, iou=0.692
restart 13: loss=632.718, iou=0.696
restart 14: loss=660.268, iou=0.687
restart 15: loss=654.997, iou=0.690
restart 16: loss=637.528, iou=0.694
restart 17: loss=662.494, iou=0.683
restart 18: loss=629.038, iou=0.700
restart 19: loss=637.497, iou=0.696
Mean pairwise distance: 25.854 ± 0.372
Min: 24.777, Max: 26.778


In [ ]:
from scipy.sparse.linalg import svds

def estimate_gauss_newton_condition(problem, x, n_power_iter=30, seed=0):
    """Estimate cond(A^T diag(h') A) via power iteration — avoids forming N^2 matrix."""
    ax = problem.conv(x)
    s = sigmoid(ax, problem.alpha, problem.c)
    sp = sigmoid_prime_from_output(s, problem.alpha)  # (H, W)

    def matvec(v_flat):
        v = v_flat.reshape(problem.image_shape)
        Av = problem.conv(v)
        weighted = sp * Av  # diag(h') A v
        return problem.conv_transpose(sp * weighted).reshape(-1)  # A^T diag(h') A v

    rng = np.random.default_rng(seed)
    n = x.size
    # Power iteration for largest eigenvalue
    q = rng.normal(size=n); q /= np.linalg.norm(q)
    for _ in range(n_power_iter):
        q = matvec(q); q /= (np.linalg.norm(q) + 1e-15)
    lam_max = float(q @ matvec(q))

    # Smallest via inverse power (shift-and-invert approximation)
    q2 = rng.normal(size=n); q2 /= np.linalg.norm(q2)
    for _ in range(n_power_iter):
        tmp = matvec(q2)
        tmp = lam_max * q2 - tmp  # (lam_max I - M) q
        q2 = tmp / (np.linalg.norm(tmp) + 1e-15)
    lam_min = lam_max - float(q2 @ matvec(q2))
    lam_min = max(lam_min, 1e-10)

    return lam_max, lam_min, lam_max / lam_min

cond_rows = []
for kname in ["identity_like", "avg_blur", "sobel_x", "laplacian", "random_norm"]:
    payload = run_single_experiment(image_shape=(32, 32), kernel_name=kname,
                                    target_name="checkerboard", alpha=10.0, c=0.5,
                                    optimizer_name="adam", optimizer_kwargs={"lr": 0.03, "steps": 200})
    lam_max, lam_min, cond = estimate_gauss_newton_condition(payload["problem"], payload["x_final"])
    cond_rows.append({"kernel": kname, "lam_max": lam_max, "lam_min": lam_min, "condition_number": cond})
    print(f"{kname}: cond={cond:.1f}")

cond_df = pd.DataFrame(cond_rows).sort_values("condition_number")
cond_df.to_csv(os.path.join(CSV_DIR, "kernel_condition_numbers.csv"), index=False)

plt.figure(figsize=(7, 4))
plt.bar(cond_df["kernel"], np.log10(cond_df["condition_number"]))
plt.ylabel("log10(condition number)")
plt.title("Gauss-Newton condition number per kernel (alpha=10)")
plt.xticks(rotation=20)
save_fig(os.path.join(FIG_DIR, "kernel_condition_numbers.pdf"))

identity_like: cond=1.8
avg_blur: cond=1.0
sobel_x: cond=1.0
laplacian: cond=1.0
random_norm: cond=1.0


In [ ]:
def print_latex_alpha_table(alpha_table_polished_df):
    """Print ready-to-paste LaTeX rows for Table 2."""
    target_alphas = [1.0, 5.0, 10.0, 20.0]
    print(r"\begin{tabular}{cccc}")
    print(r"\toprule")
    print(r"$\alpha$ & Loss (mean $\pm$ std) & IoU (mean $\pm$ std) & Best optimizer \\")
    print(r"\midrule")
    for alpha in target_alphas:
        sub = alpha_table_polished_df[alpha_table_polished_df["alpha"] == alpha]
        best_row = sub.loc[sub["loss_final_mean"].idxmin()]
        best_opt = best_row["optimizer"]
        loss_m = best_row["loss_final_mean"]; loss_s = best_row["loss_final_std"]
        iou_m = best_row["output_iou_final_mean"]; iou_s = best_row["output_iou_final_std"]
        print(f"{alpha:.1f} & {loss_m:.3f} $\\pm$ {loss_s:.3f} & {iou_m:.3f} $\\pm$ {iou_s:.3f} & {best_opt} \\\\")
    print(r"\bottomrule")
    print(r"\end{tabular}")

print_latex_alpha_table(alpha_table_polished)

\begin{tabular}{cccc}
\toprule
$\alpha$ & Loss (mean $\pm$ std) & IoU (mean $\pm$ std) & Best optimizer \\
\midrule
1.0 & 493.792 $\pm$ 11.485 & 0.834 $\pm$ 0.009 & lbfgsb \\
5.0 & 344.187 $\pm$ 11.618 & 0.836 $\pm$ 0.006 & adam \\
10.0 & 488.052 $\pm$ 9.103 & 0.762 $\pm$ 0.004 & adam \\
20.0 & 646.307 $\pm$ 11.351 & 0.691 $\pm$ 0.005 & adam \\
\bottomrule
\end{tabular}


In [ ]:
from google.colab import files
files.download("/content/fixed_cnn_inverse_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>